# CFPS · Publication-ready Plate Decision Lab

**Purpose:** Combine the original GP-informed submitted-recipe designations and a model-free reagent-volume distance strategy under exact physical-well and replicate constraints, then build auditable **provisional** maps for sfGFP, PETase and Reteplase. 

**Plate evidence:** inspected Ginkgo CFPS example plate JSONs use 16×24 wells, reserve columns 1–4, expose columns 5–24 (320 experimental wells), and set `replicate_factor=4`, i.e. **80 independent formulations with four technical replicates**. These assumptions are configurable, and actual laboratory approval is still required. The three maps are visualization/planning artifacts, **not robot execution instructions**. Ginkgo may randomize sample placement in execution.

**Sources:** `published_snapshot` loads the original GP-annotated `clusters_designs.json` (immutable input). `pocketbase_snapshot` loads a private authorized raw PocketBase JSON snapshot; `pocketbase_live` fetches `cfps_designs` through a supplied authorized PocketBase token or password. PocketBase-only data has *no* published GP roles or forecasts: its allowed GP share is zero. No GP forecast or cluster is fabricated. The notebook records SHA-256 hashes.

**Contributions:** Original GP-informed forecasts/group designations: Clinton Monteiro; model-free geometry/decision framework: Kourosh Afshinjoo; application infrastructure: Ronan Chance. Methodological terminology in the implementation remains objective.

**Privacy:** all participant-level outputs, the HTML with private data loader and private JSON, are kept in Colab's private runtime and downloaded as a ZIP. **Do not commit private ZIP contents or generated `decision_input_private.json` to Git.**

In [ ]:
%pip -q install numpy scipy requests nbformat pytest
from pathlib import Path
import os, json, hashlib, datetime, getpass, zipfile
PRIVATE = Path('/content/cfps_decision_private')
PRIVATE.mkdir(parents=True, exist_ok=True)
ROOT = Path('/content/cfps_decision_runtime')
ROOT.mkdir(parents=True, exist_ok=True)


## 1 · Change inputs and strategy parameters here

`gp_wells=None` means the percentage slider controls the final quota. `gp_wells=160` means *exactly* 160 physical wells reserved for GP-informed selections. The remaining experimental wells are filled model-free. Replicate factor must divide 320, and explicit GP wells must divide the replicate factor.

The **published-designations** policy uses the original `round1Role` ordering for GP candidates; if its quota requires more candidates than annotated original roles, any additional selections are ranked by GP forecasts and **explicitly flagged**. The `forecast_ranked` policy ignores role priority and uses rank-normalized forecast criteria. These criteria are not calibrated biological utilities.

In [ ]:
#@title Input source and experimental design
INPUT_SOURCE = 'published_snapshot'  # 'published_snapshot' | 'pocketbase_snapshot' | 'pocketbase_live'
INPUT_PATH = '/content/clusters_designs.json'  # or PocketBase JSON path
PB_URL = 'https://db.rcdonovan.com'  # only used for pocketbase_live
PB_AUTH_COLLECTION = '_superusers'  # must match YOUR authorized deployment
REPLICATE_FACTOR = 4  # inspected Ginkgo example default (80 unique / protein plate)
GP_SHARE_PERCENT = 50  # set 0 for PocketBase-only input
GP_WELLS = None  # integer physical wells overrides GP_SHARE_PERCENT (e.g., 160)
GP_POLICY = 'published_designations'  # or 'forecast_ranked'
MODEL_FREE_CRITERION = 'maximin'  # or 'facility_location' / 'balanced'
BALANCED_MAXIMIN_WEIGHT = 0.5
GP_YIELD_WEIGHT = 0.60
GP_COST_WEIGHT = 0.25
GP_UNCERTAINTY_WEIGHT = 0.15


## 2 · Self-contained implementation

The code below writes unmodified source modules bundled into this notebook to a temporary Colab package and imports them for this session. **No GitHub checkout or external source download is required.** Do not edit generated code in the output cells; edit the mirrored source module and rebuild the notebook in the repository.

In [ ]:
rel='analysis/__init__.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Analysis modules for reproducible HTGAA CFPS research workflows."""\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
rel='analysis/cfps_selection/__init__.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Pre-run CFPS experiment-selection audit, coverage, and candidate selectors."""\nfrom .selection_framework import (\n    AuditResult, audit_gp_model_snapshot, audit_gp_model_link_matrices, changed_reagents, canonical_recipe,\n    read_gp_model_file, coverage_metrics, suggest_portfolio, write_selection_overlay,\n)\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
rel='analysis/cfps_selection/selection_framework.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Reproducible, pre-experimental CFPS candidate audit and portfolio diagnostics.\n\nInputs: the dated GP-informed clusters_designs.json export (no measurements required).\nAll output is a description of submitted recipes or PRE-RUN model predictions.\nNo code here retrains the published GP or estimates empirical response correlation.\n\nThe same source is embedded *verbatim* into the independently executable Colab notebook.\nVersion: 1.1.0 (2026-10-27).\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport gzip\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial.distance import cdist\n\nVERSION = "1.1.0"\nWORKING_QUOTA = 320  # NOT verified final competition rules\nREACTION_NL = 20_000.0\nEXCLUDED_BALANCERS = frozenset({"nuclease_free_water"})\nBASELINE_ROLES = frozenset({"first run", "check run", "independent run", "sweep run"})\nTARGETS = ("sfGFP", "PETase", "Reteplase")\n\n\ndef canonical_recipe(recipe: Mapping[str, Any], *, exclude_water: bool = True) -> tuple:\n    """Immutable recipe identity; zeros and absent keys are equivalent.\n\n    This identifies exact *dispensing recipes*, NOT chemical equivalence when stock\n    concentrations or identities differ. Does not compare across protein targets.\n    """\n    if not isinstance(recipe, Mapping):\n        raise TypeError("recipe must be a mapping")\n    out = []\n    for key, raw in recipe.items():\n        if exclude_water and key in EXCLUDED_BALANCERS:\n            continue\n        if isinstance(raw, bool):\n            raise ValueError(f"Boolean volume for {key}")\n        try:\n            vol = float(raw)\n        except (TypeError, ValueError) as exc:\n            raise ValueError(f"Non-numeric volume for {key}: {raw!r}") from exc\n        if not np.isfinite(vol) or vol < 0:\n            raise ValueError(f"Invalid volume for {key}: {vol!r}")\n        if vol != 0:\n            out.append((str(key), round(vol, 9)))\n    return tuple(sorted(out))\n\n\ndef signature_except(recipe: Mapping[str, Any], factor: str) -> tuple:\n    return tuple((k, v) for k, v in canonical_recipe(recipe) if k != factor)\n\n\ndef recipe_factor_value(recipe: Mapping[str, Any], factor: str) -> float:\n    return float(recipe.get(factor, 0))\n\n\ndef changed_reagents(recipe_a: Mapping[str, Any], recipe_b: Mapping[str, Any]) -> list[str]:\n    a, b = dict(canonical_recipe(recipe_a)), dict(canonical_recipe(recipe_b))\n    return [key for key in sorted(a.keys() | b.keys()) if not np.isclose(a.get(key, 0), b.get(key, 0), rtol=0, atol=1e-8)]\n\n\ndef input_sha256(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef _factor_key_from_display(ingredient: str, record: Mapping[str, Any]) -> str:\n    """Resolve check\'s display label to stored reagent key using check\'s own data.\n\n    Prefer exact `group.checks.ingredient` mapped through displayed names from\n    vsGinkgoPlates or sweeps; handle common aliases for non-tested reagents.\n    Fail closed when unknown, instead of guessing from textual similarity.\n    """\n    common = {\n        "DsbC (E. coli, 25.5 kDa)": "dsbc_ecoli",\n        "PEG-8000": "peg_8000",\n        "Magnesium Glutamate": "magnesium_glutamate",\n        "HEPES-KOH pH 7.5": "hepes_koh",\n        "Coenzyme A": "coa",\n        "PEP mono (0.1M stock)": "pep_mono",\n        "17 Amino Acid Mix": "amino_acid_mix_17",\n        "Nicotinamide": "nicotinamide",\n        "Glucose": "glucose",\n        "Folinic Acid": "folinic_acid",\n        "Potassium Glutamate": "potassium_glutamate",\n        "Potassium Oxalate": "potassium_oxalate",\n        "Ammonium Glutamate": "ammonium_glutamate",\n        "Reduced Glutathione (GSH)": "reduced_glutathione",\n        "Oxidized Glutathione (GSSG)": "oxidized_glutathione",\n        "Maltodextrin 17": "maltodextrin_17",\n        "Spermidine": "spermidine",\n        "Tyrosine pH 12": "tyrosine",\n        "Potassium phosphate 1.6:1 dibasic:monobasic (molar ratio)": "kpo_dibasic_mix",\n        "Potassium phosphate 1.6:1 monobasic:dibasic (molar ratio)": "kpo_monobasic_mix",\n        "Putrescine Dihydrochloride": "putrescine",\n    }\n    for category in ("neverOnAPlate", "pastThePlatesRange"):\n        for d in record.get("vsGinkgoPlates", {}).get(category, []):\n            if d.get("name") == ingredient:\n                return str(d["reagent"])\n    if ingredient in common:\n        return common[ingredient]\n    slug = ingredient.lower().replace("-", "_").replace(" ", "_")\n    if slug in record.get("recipeNl", {}):\n        return slug\n    for reagent in record.get("recipeNl", {}):\n        if str(reagent).casefold() == ingredient.casefold():\n            return reagent\n    return ""  # Caller must flag unresolved; no inferential fallback.\n\n\ndef _validated_records(data: Mapping[str, Any]) -> tuple[dict[str, dict[str, dict]], list[dict]]:\n    if not isinstance(data, Mapping) or not isinstance(data.get("proteins"), Mapping):\n        raise ValueError("Expected published clusters_designs.json with proteins mapping")\n    by_target = {}\n    rows = []\n    global_seen = set()\n    for target in sorted(data["proteins"]):\n        block = data["proteins"][target]\n        designs = block.get("designs")\n        if not isinstance(designs, list):\n            raise ValueError(f"Missing designs list for {target}")\n        lookup = {}\n        for design in designs:\n            rid = str(design["id"])\n            if rid in lookup or rid in global_seen:\n                raise ValueError(f"Nonunique design ID: {rid}")\n            global_seen.add(rid)\n            if design.get("protein") != target:\n                raise ValueError(f"Design {rid} target mismatch")\n            recipe = design.get("recipeNl")\n            key = canonical_recipe(recipe)\n            total = sum(float(v) for v in recipe.values())\n            lookup[rid] = design\n            rows.append({\n                "protein": target, "id": rid,\n                "sampleId": design.get("sampleId"), "author": design.get("author"),\n                "round1Role": design.get("round1Role"),\n                "selectedProposed": bool(design.get("round1Role") in BASELINE_ROLES),\n                "sameRecipeAs": design.get("sameRecipeAs"),\n                "group": (design.get("group") or {}).get("letter"),\n                "recipe_key": key,\n                "recipe_hash": hashlib.sha256(json.dumps(key).encode()).hexdigest(),\n                "total_nL": total,\n                "total_valid": abs(total - REACTION_NL) < 1e-5,\n                "n_active_reagents": len(key),\n                "pBeat_model": (design.get("forecast") or {}).get("pBeat"),\n                "lnCostPerGram_model": (design.get("forecast") or {}).get("lnSpecCost"),\n            })\n        by_target[target] = lookup\n    return by_target, rows\n\n\ndef _sweep_rows(data: Mapping[str, Any], by_target: dict, selected_ids: dict) -> list[dict]:\n    result = []\n    for target in sorted(by_target):\n        lookup = by_target[target]\n        for ingredient_block in data["proteins"][target].get("sweeps", []):\n            factor = ingredient_block["reagent"]\n            for family_index, sweep in enumerate(ingredient_block.get("sweeps", [])):\n                to_run = sweep.get("toRun") or []\n                if not to_run:\n                    continue\n                ids = [x.get("id") if isinstance(x, Mapping) else x for x in to_run]\n                ids = [str(x) for x in ids]\n                missing = [rid for rid in ids if rid not in lookup]\n                if missing:\n                    raise ValueError(f"Unknown IDs in sweep {target}/{factor}: {missing}")\n                unique_selected = {}\n                background = set()\n                for rid in ids:\n                    if rid not in selected_ids[target]:\n                        continue\n                    design = lookup[rid]\n                    unique_selected[canonical_recipe(design["recipeNl"])] = design\n                    background.add(signature_except(design["recipeNl"], factor))\n                levels = sorted({float(r["recipeNl"].get(factor, 0)) for r in unique_selected.values()})\n                result.append({\n                    "protein": target, "factor": factor, "family_index": family_index,\n                    "recommendedByModel": bool(ingredient_block.get("recommended")),\n                    "n_toRun_ids": len(ids), "n_toRun_unique": len(unique_selected),\n                    "n_selected_levels": len(levels), "selected_levels_nL": ";".join(map(lambda x: str(int(x)) if x.is_integer() else str(x), levels)),\n                    "background_signature_count": len(background),\n                    "clean_single_factor_sweep": len(background) == 1,\n                    "at_least_three_levels": len(levels) >= 3,\n                    "run_ids": ";".join(ids),\n                })\n    return result\n\n\ndef _cluster_checks(data: Mapping[str, Any], by_target: dict, selected_ids: dict) -> tuple[list[dict], list[dict]]:\n    checks, additions = [], []\n    for target in sorted(by_target):\n        lookup = by_target[target]\n        for group in data["proteins"][target].get("groups", []):\n            base_id = str(group["firstRun"])\n            if base_id not in lookup:\n                raise ValueError(f"Unknown first run: {base_id}")\n            base = lookup[base_id]\n            for check in group.get("checkRuns", []):\n                check_id = str(check["id"])\n                if check_id not in lookup:\n                    raise ValueError(f"Unknown check run: {check_id}")\n                design = lookup[check_id]\n                factor = _factor_key_from_display(check["ingredient"], design)\n                if not factor:\n                    # Try first-run fields, as named ingredient might be absent in check\n                    factor = _factor_key_from_display(check["ingredient"], base)\n                changed = changed_reagents(base["recipeNl"], design["recipeNl"])\n                named_changed = factor in changed if factor else False\n                exact_selected = []\n                exact_unselected = []\n                if factor:\n                    sig = signature_except(design["recipeNl"], factor)\n                    for other_id, other in lookup.items():\n                        if other_id == check_id:\n                            continue\n                        if signature_except(other["recipeNl"], factor) != sig:\n                            continue\n                        if np.isclose(recipe_factor_value(other["recipeNl"], factor), recipe_factor_value(design["recipeNl"], factor)):\n                            continue\n                        (exact_selected if other_id in selected_ids[target] else exact_unselected).append(other_id)\n                exact_selected = sorted(exact_selected)\n                exact_unselected = sorted(exact_unselected)\n                checks.append({\n                    "protein": target, "group": group["letter"],\n                    "firstRunId": base_id, "checkRunId": check_id,\n                    "namedFactorLabel": check["ingredient"], "namedFactor": factor,\n                    "named_factor_changed": named_changed,\n                    "changed_factor_count": len(changed),\n                    "changed_factors": ";".join(changed),\n                    "exact_one_factor_with_first_run": len(changed) == 1 and named_changed,\n                    "any_selected_exact_counterpart": bool(exact_selected),\n                    "any_unselected_exact_counterpart": bool(exact_unselected),\n                    "model_link_with_first": (design.get("group") or {}).get("linkWithFirstRun"),\n                    "model_new_information_pct": check.get("newInformation"),\n                    "model_price_only": check.get("priceOnly"),\n                })\n                if not exact_selected and exact_unselected and factor:\n                    # Stable deterministic candidate: prefer the largest dose contrast, then lowest ID.\n                    own_level = recipe_factor_value(design["recipeNl"], factor)\n                    chosen = sorted(exact_unselected, key=lambda rid: (-abs(recipe_factor_value(lookup[rid]["recipeNl"], factor) - own_level), rid))[0]\n                    additions.append({\n                        "protein": target, "group": group["letter"],\n                        "checkRunId": check_id, "namedFactor": factor,\n                        "checkFactorNl": own_level, "candidateId": chosen,\n                        "candidateFactorNl": recipe_factor_value(lookup[chosen]["recipeNl"], factor),\n                        "is_exact_one_factor_against_check": True,\n                    })\n    return checks, additions\n\n\n@dataclass\nclass AuditResult:\n    summary: dict\n    candidates: pd.DataFrame\n    duplicates: pd.DataFrame\n    check_runs: pd.DataFrame\n    sweep_families: pd.DataFrame\n    additional_contrasts: pd.DataFrame\n    records: dict[str, dict[str, dict]]\n\n    def save(self, out: str | Path, *, input_paths: Sequence[str | Path] = ()) -> dict:\n        """Write private row-level CSVs and provenance manifest.\n\n        The `publish_row_level` flag requires explicit opt-in; caller controls actual\n        directory permissions. No code uploads or pushes files to Git automatically.\n        """\n        out = Path(out)\n        out.mkdir(parents=True, exist_ok=True)\n        exports = {\n            "candidate_audit.csv": self.candidates.drop(columns=["recipe_key"], errors="ignore"),\n            "duplicate_formulations.csv": self.duplicates,\n            "cluster_check_audit.csv": self.check_runs,\n            "sweep_family_audit.csv": self.sweep_families,\n            "additional_exact_contrasts.csv": self.additional_contrasts,\n        }\n        for filename, frame in exports.items():\n            frame.to_csv(out / filename, index=False)\n        with open(out / "audit_summary.json", "w") as fh:\n            json.dump(self.summary, fh, indent=2, sort_keys=True, allow_nan=False)\n        manifest = {\n            "schema": "cfps-selection-audit/v1", "analysis_version": VERSION,\n            "input_sha256": {Path(p).name: input_sha256(p) for p in input_paths},\n            "working_quota": self.summary["working_quota_per_protein"],\n            "python": __import__("sys").version.split()[0],\n            "numpy": np.__version__, "pandas": pd.__version__,\n            "scipy": __import__("scipy").__version__,\n            "parameters": {\n                "reaction_nL": REACTION_NL,\n                "baseline_roles": sorted(BASELINE_ROLES),\n                "duplicate_recipe_excludes": sorted(EXCLUDED_BALANCERS),\n                "recipe_volume_unit": "nL (stock dispensing volumes, not molarity)",\n                "selected_unique_rule": "one exact dispensing formulation per target",\n            },\n            "files": sorted(exports) + ["audit_summary.json"],\n            "classification": "row-level competition data; private until approved",\n            "publication_review_required": True,\n        }\n        (out / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\\n", encoding="utf-8")\n        return manifest\n\n\ndef audit_gp_model_snapshot(data: Mapping[str, Any], *, working_quota: int = WORKING_QUOTA) -> AuditResult:\n    """Deterministic end-to-end audit; never modifies raw data or model forecasts."""\n    if int(working_quota) < 1:\n        raise ValueError("working_quota must be positive")\n    by_target, rows = _validated_records(data)\n    candidates = pd.DataFrame(rows).sort_values(["protein", "id"], kind="stable").reset_index(drop=True)\n    # Internal exact-key maps; no cross-target merging.\n    selected_ids = {}\n    duplicates = []\n    summary_proteins = {}\n    for target in sorted(by_target):\n        part = candidates.loc[candidates["protein"] == target].copy()\n        selected = part.loc[part["selectedProposed"]]\n        all_groups = defaultdict(list)\n        for r in part.itertuples(index=False):\n            all_groups[r.recipe_key].append(r.id)\n        selected_groups = defaultdict(list)\n        for r in selected.itertuples(index=False):\n            selected_groups[r.recipe_key].append(r.id)\n        selected_ids[target] = set(selected["id"])\n        for signature, ids in all_groups.items():\n            if len(ids) > 1:\n                duplicates.append({\n                    "protein": target, "count": len(ids), "ids": ";".join(sorted(ids)),\n                    "canonical_signature_sha256": hashlib.sha256(repr(signature).encode()).hexdigest(),\n                    "selected_proposed_count": sum(rid in selected_ids[target] for rid in ids),\n                })\n        block = data["proteins"][target]\n        untested_groups = sum(bool(g.get("linkFromSharedUntested", 0) > 0) for g in block.get("groups", []))\n        summary_proteins[target] = {\n            "candidateDesigns": int(len(part)),\n            "distinctFormulations": int(len(all_groups)),\n            "duplicateExcess": int(len(part) - len(all_groups)),\n            "selectedRoleRecords": int(len(selected)),\n            "selectedUniqueFormulations": int(len(selected_groups)),\n            "selectedDuplicateExcess": int(len(selected) - len(selected_groups)),\n            "gapToWorkingQuota": int(working_quota - len(selected_groups)),\n            "roleBreakdownUniqueDesignIDs": {str(k): int(v) for k,v in selected["round1Role"].value_counts().sort_index().items()},\n            "groups": len(block.get("groups", [])),\n            "groupsWithSharedUntested": int(untested_groups),\n            "authorsAll": int(part["author"].nunique(dropna=True)),\n            "authorsSelected": int(selected["author"].nunique(dropna=True)),\n            "volumeValidRecords": int(part["total_valid"].sum()),\n            "sourceTotals": block.get("totals", {}),\n        }\n    checks, add = _cluster_checks(data, by_target, selected_ids)\n    sweeps = _sweep_rows(data, by_target, selected_ids)\n    check_cols = ["protein", "group", "firstRunId", "checkRunId", "namedFactorLabel", "namedFactor", "named_factor_changed", "changed_factor_count", "changed_factors", "exact_one_factor_with_first_run", "any_selected_exact_counterpart", "any_unselected_exact_counterpart", "model_link_with_first", "model_new_information_pct", "model_price_only"]\n    sweep_cols = ["protein", "factor", "family_index", "recommendedByModel", "n_toRun_ids", "n_toRun_unique", "n_selected_levels", "selected_levels_nL", "background_signature_count", "clean_single_factor_sweep", "at_least_three_levels", "run_ids"]\n    add_cols = ["protein", "group", "checkRunId", "namedFactor", "checkFactorNl", "candidateId", "candidateFactorNl", "is_exact_one_factor_against_check"]\n    checks_df = pd.DataFrame(checks,columns=check_cols).sort_values(["protein", "group", "checkRunId"]).reset_index(drop=True)\n    sweeps_df = pd.DataFrame(sweeps,columns=sweep_cols).sort_values(["protein", "factor", "family_index"]).reset_index(drop=True)\n    add_df = pd.DataFrame(add,columns=add_cols).sort_values(["protein", "group", "checkRunId"]).reset_index(drop=True)\n    for target in summary_proteins:\n        c = checks_df[checks_df["protein"] == target]\n        s = sweeps_df[sweeps_df["protein"] == target]\n        a = add_df[add_df["protein"] == target]\n        summary_proteins[target].update({\n            "clusterChecks": int(len(c)),\n            "exactOneFactorChecksVsFirst": int(c["exact_one_factor_with_first_run"].sum()),\n            "multifactorChecksVsFirst": int((c["changed_factor_count"] > 1).sum()),\n            "unresolvedNamedFactorChecks": int((c["namedFactor"] == "").sum()),\n            "newExactCounterpartsForChecks": int(len(a)),\n            "listedSweepFamiliesWithRuns": int(len(s)),\n            "cleanSweepFamilies": int(s["clean_single_factor_sweep"].sum()),\n            "sweepFamiliesWithThreeOrMoreLevels": int(s["at_least_three_levels"].sum()),\n        })\n    duplicate_df = pd.DataFrame(duplicates).sort_values(["protein", "canonical_signature_sha256"]).reset_index(drop=True)\n    summary = {\n        "analysis_version": VERSION,\n        "source_built_utc": data.get("built"),\n        "scope": "audit of published GP-informed proposed runs, not measured results or final selection",\n        "working_quota_per_protein": int(working_quota),\n        "proteins": summary_proteins,\n        "interpretation": {\n            "model_based": "forecast, expected links and LHS sensitivities are pre-run model products",\n            "model_free": "recipe identities, exact contrasts, sweep completeness, distance/coverage",\n            "price_per_gram": "model forecast; not validated observed cost per gram",\n            "experimental_info_gain": "NOT calculated: model newInformation is a heuristic",\n        },\n    }\n    return AuditResult(summary, candidates, duplicate_df, checks_df, sweeps_df, add_df, by_target)\n\n\ndef read_gp_model_file(path: str | Path) -> dict:\n    with open(path, "rb") as fh:\n        return json.load(fh)\n\n\ndef _variable_recipe_matrix(rows: list[dict], *, feature_mode: str = "scaled_volume", presence_weight: float = 0.35) -> tuple[np.ndarray, list[str]]:\n    """Candidate rows -> scaled recipe matrix; no model predictions involved.\n\n    Excludes water and columns invariant in this protein. For concentrations,\n    use stock nL only as a per-reagent within-stock proxy, NOT chemical molarity.\n    Normalize by max observed nL (>=1) to make each ingredient dimension comparable.\n    """\n    if feature_mode not in {"scaled_volume", "presence", "scaled_plus_presence"}:\n        raise ValueError("Invalid feature_mode")\n    all_recipes = [r["recipeNl"] for r in rows]\n    keys = sorted({k for r in all_recipes for k in r if k not in EXCLUDED_BALANCERS})\n    A = np.asarray([[float(rec.get(k, 0)) for k in keys] for rec in all_recipes], dtype=float)\n    var = np.ptp(A, axis=0) > 1e-9\n    A = A[:, var]\n    keys = [k for k, keep in zip(keys, var) if keep]\n    if not keys:\n        raise ValueError("No variable reagent dimensions")\n    scales = np.maximum(np.max(A, axis=0), 1)\n    Z = A / scales\n    if feature_mode == "presence":\n        return (A > 0).astype(float), keys\n    if feature_mode == "scaled_plus_presence":\n        Z = np.column_stack([Z, (A > 0).astype(float)*math.sqrt(presence_weight)])\n        return Z, keys + ["presence:"+k for k in keys]\n    return Z, keys\n\n\ndef coverage_metrics(candidate_rows: list[dict], selected_ids: set[str], *, metric: str = "euclidean") -> dict:\n    """Model-free dispersion/coverage diagnostics of an existing-recipe portfolio.\n\n    A candidate-population metric: no claim of uniformity in unobserved chemical space.\n    """\n    selected_indices = [i for i,r in enumerate(candidate_rows) if r["id"] in selected_ids]\n    if not selected_indices:\n        raise ValueError("No selected IDs in candidate population")\n    Z, keys = _variable_recipe_matrix(candidate_rows, feature_mode="scaled_plus_presence")\n    ds = cdist(Z, Z[selected_indices], metric=metric)\n    nearest = ds.min(axis=1)\n    # Unique physical formulations, not student submission records, are the\n    # scientific sampling units. Equal-weight each distinct recipe.\n    signatures = [canonical_recipe(r["recipeNl"]) for r in candidate_rows]\n    representative_indices = list({sig:i for i,sig in reversed(list(enumerate(signatures)))}.values())\n    unique_nearest = nearest[representative_indices]\n    factors = {}\n    for k in sorted({k for r in candidate_rows for k in r["recipeNl"] if k not in EXCLUDED_BALANCERS}):\n        all_values = np.array([float(r["recipeNl"].get(k,0)) for r in candidate_rows])\n        sel_values = all_values[selected_indices]\n        unique_positive = sorted(set(all_values[all_values>0]))\n        # Bins drawn from candidate quantiles: a coverage *diagnostic*, not a perfect LHS.\n        if len(unique_positive) >= 2:\n            bounds = np.unique(np.quantile(unique_positive, np.linspace(0,1,6)))\n            pos_all = np.digitize(all_values[all_values>0], bounds[1:-1])\n            pos_sel = np.digitize(sel_values[sel_values>0], bounds[1:-1])\n            all_bins = set(pos_all)\n            sel_bins = set(pos_sel)\n            factor_coverage = len(sel_bins & all_bins)/len(all_bins) if all_bins else 1.0\n        else:\n            factor_coverage = 1.0 if (len(unique_positive)==0 or (sel_values>0).any()) else 0.0\n        factors[k] = {"candidate_bin_coverage": round(float(factor_coverage),6),\n                      "has_absent": bool((sel_values==0).any()),\n                      "has_present": bool((sel_values>0).any())}\n    return {\n        "n_selected_ids": len(selected_indices),\n        "n_distinct_candidate_formulations": len(representative_indices),\n        "feature_mode": "scaled_plus_presence", "n_features": Z.shape[1],\n        "mean_candidate_to_selected_distance": float(unique_nearest.mean()),\n        "p90_candidate_to_selected_distance": float(np.quantile(unique_nearest, 0.90)),\n        "max_candidate_to_selected_distance": float(unique_nearest.max()),\n        "factor_coverage": factors,\n    }\n\n\ndef suggest_portfolio(\n    candidate_rows: list[dict], *, quota: int = WORKING_QUOTA,\n    mode: str = "hybrid", objective: str = "facility",\n    seed_ids: Sequence[str] = (),\n    exact_unique: bool = True,\n) -> dict:\n    """Deterministic proposed IDs. No new recipes; no GP calls.\n\n    - \'gp_model_baseline\': return the original GP-informed proposed selection *without padding*.\n    - \'hybrid\': lock the published GP-informed baseline plus explicitly given seeds, fill to quota.\n    - \'model_free\': ignore model-based roles/predictions, use only seeds and geometry.\n\n    A *diagnostic selection* only. Does not guarantee experimental contrast quality,\n    student fairness, batch feasibility, or a scientifically optimal design.\n    """\n    if mode not in {"gp_model_baseline", "hybrid", "model_free"}:\n        raise ValueError("mode must be gp_model_baseline/hybrid/model_free")\n    if objective not in {"facility", "maximin"}:\n        raise ValueError("objective must be facility/maximin")\n    if quota < 1:\n        raise ValueError("quota must be positive")\n    ordered = sorted(candidate_rows, key=lambda r:r["id"])\n    idmap = {r["id"]:i for i,r in enumerate(ordered)}\n    if len(idmap) != len(ordered):\n        raise ValueError("Non-unique IDs")\n    # Published export labels include multiple IDs for the *same* dispensing formulation.\n    # Physical baseline counts are unique chemical recipes; preserve provenance via\n    # duplicate-group output, and prefer a first run, check, sweep, independent.\n    priority = {"first run": 0, "check run": 1, "sweep run": 2, "independent run": 3}\n    reps = {}\n    for r in sorted(ordered, key=lambda x:(priority.get(x["round1Role"],99),x["id"])):\n        if r["round1Role"] in BASELINE_ROLES:\n            reps.setdefault(canonical_recipe(r["recipeNl"]), r["id"])\n    baseline = set(reps.values())\n    if mode == "gp_model_baseline":\n        return {"mode":mode,"objective": "baseline", "ids": sorted(baseline),\n                "count":len(baseline),"quota_gap":quota-len(baseline)}\n    selected = set(seed_ids)\n    if unknown := selected-set(idmap):\n        raise ValueError(f"Unknown seeds: {sorted(unknown)}")\n    if mode == "hybrid":\n        already_represented = {canonical_recipe(ordered[idmap[rid]]["recipeNl"]) for rid in selected}\n        selected.update(rid for signature,rid in reps.items() if signature not in already_represented)\n    if len(selected) > quota:\n        raise ValueError(f"Selected seeds/baseline exceed quota {quota}")\n    # Identity buckets: even when records have separate IDs, avoid paying twice for same recipe.\n    keys = [canonical_recipe(r["recipeNl"]) for r in ordered]\n    represented = {keys[idmap[rid]] for rid in selected}\n    if exact_unique and len(represented) < len(selected):\n        raise ValueError("Seed/baseline contains chemically identical formulations")\n    allowed = [i for i, r in enumerate(ordered) if (not exact_unique or keys[i] not in represented)]\n    Z, feature_names = _variable_recipe_matrix(ordered, feature_mode="scaled_plus_presence")\n    dist = cdist(Z, Z, metric="euclidean")\n    # Equal weight to distinct formulations; otherwise students submitting\n    # identical recipes many times would bias coverage optimization.\n    signature_count = Counter(keys)\n    weights = np.asarray([1.0 / signature_count[k] for k in keys])\n    weights = weights / weights.sum()\n    # Initial seed-free selection: medoid minimizes total candidate distance for facility,\n    # maximum total distance for maximin (farthest extremum; documented heuristic).\n    best_distance = np.min(dist[:,[idmap[i] for i in selected]], axis=1) if selected else None\n    while len(selected)<quota and allowed:\n        if best_distance is None:\n            scores = weights @ dist\n            pick = min(allowed, key=lambda i:(scores[i],ordered[i]["id"])) if objective=="facility" else max(allowed,key=lambda i:(scores[i],ordered[i]["id"]))\n        elif objective=="maximin":\n            pick = max(allowed, key=lambda i:(float(best_distance[i]),ordered[i]["id"]))\n        else:\n            # Weighted candidate coverage: minimize sum of nearest distances.\n            pick = min(allowed,key=lambda i:(float(weights @ np.minimum(best_distance,dist[:,i])),ordered[i]["id"]))\n        chosen_id = ordered[pick]["id"]\n        selected.add(chosen_id)\n        represented.add(keys[pick])\n        best_distance = dist[:,pick].copy() if best_distance is None else np.minimum(best_distance,dist[:,pick])\n        allowed = [i for i in allowed if (i != pick and (not exact_unique or keys[i] not in represented))]\n    return {"mode":mode,"objective":objective,"ids":sorted(selected),\n            "count":len(selected), "quota_gap":quota-len(selected),\n            "n_features":len(feature_names),\n            "coverage_mean":float(weights @ best_distance) if best_distance is not None else None,\n            "selection_status":"PROVISIONAL: geometry-only fill; controls/contrasts/fairness require separate review"}\n\n\ndef write_selection_overlay(\n    audit: AuditResult, portfolios: Mapping[str, Mapping[str, dict]], path: str | Path,\n    *, include_private_ids: bool = False,\n) -> None:\n    """Precomputed JSON for an opt-in browser visualization; author names omitted.\n\n    ID-level overlay remains private by default; explicit approval before publishing.\n    """\n    result = {"schema":"cfps-selection-overlay/v1", "audit_version":VERSION,\n              "snapshot_built":audit.summary["source_built_utc"], "targets":{}}\n    for target in sorted(audit.records):\n        port = portfolios.get(target, {})\n        pitems = {}\n        for label, d in port.items():\n            pitems[label] = {k:v for k,v in d.items() if k not in {"ids"}}\n            if include_private_ids:\n                pitems[label]["ids"] = d.get("ids",[])\n        result["targets"][target] = {\n            "audit": {k:v for k,v in audit.summary["proteins"][target].items() if k!="sourceTotals"},\n            "portfolios":pitems,\n        }\n    Path(path).write_text(json.dumps(result,indent=2,sort_keys=True,allow_nan=False)+"\\n",encoding="utf-8")\n\n\ndef audit_gp_model_link_matrices(\n    links: Mapping[str, Any] | str | Path, *, design_audit: AuditResult | None = None\n) -> dict:\n    """Audit *published/displayed* link matrices (not estimated GP covariance).\n\n    For cross-snapshot validity, check matching row counts separately; no IDs are\n    inferred from the rendered `names` strings. A rounded correlation-like matrix\n    with negative eigenvalues should NOT be used as a GP covariance without an\n    explicitly justified PSD repair and model verification.\n    """\n    if isinstance(links, (str, Path)):\n        p = Path(links)\n        with (gzip.open(p, "rt", encoding="utf-8") if p.suffix == ".gz" else p.open("r", encoding="utf-8")) as f:\n            data = json.load(f)\n    else:\n        data = links\n    if not isinstance(data, Mapping) or "tips" not in data:\n        raise ValueError("Expected published clusters.json.gz with \'tips\'")\n    result = {"kind":"DISPLAY link-matrix audit, not biological outcome correlation",\n              "built":data.get("built"), "targets":{}}\n    names = {"sfgfp":"sfGFP", "petase":"PETase", "reteplase":"Reteplase"}\n    for key, tip in sorted(data["tips"].items()):\n        A = np.asarray(tip["C"],dtype=float)\n        if A.ndim != 2 or A.shape[0] != A.shape[1] or not np.isfinite(A).all():\n            raise ValueError(f"Invalid link matrix for {key}")\n        n=A.shape[0]\n        symmetric_max=float(np.max(np.abs(A-A.T)))\n        B=(A+A.T)/2.0\n        eig=np.linalg.eigvalsh(B)\n        target=names.get(key,key)\n        expected = (design_audit.summary["proteins"].get(target) or {}).get("candidateDesigns") if design_audit else None\n        result["targets"][target] = {\n            "n_rows":int(n),"n_names":len(tip.get("names",[])),\n            "candidate_count_matches":None if expected is None else bool(n==expected),\n            "entry_min":float(A.min()),"entry_max":float(A.max()),\n            "diagonal_max_abs_error_from_one":float(np.max(np.abs(np.diag(A)-1))),\n            "max_absolute_asymmetry":symmetric_max,\n            "min_eigenvalue_of_symmetrized_DISPLAY_matrix":float(eig[0]),\n            "negative_eigenvalue_count_below_minus_1e_6":int((eig < -1e-6).sum()),\n            "PSD_as_stored_tolerance_1e_6": bool(eig[0]>=-1e-6),\n            "interpretation":"Display values rounded; not independently verified GP covariance",\n        }\n    if design_audit is not None:\n        check=design_audit.check_runs.dropna(subset=["model_link_with_first","model_new_information_pct"])\n        if len(check):\n            v=np.asarray(check["model_link_with_first"],dtype=float)\n            scores=np.asarray(check["model_new_information_pct"],dtype=float)\n            deviations=np.abs(scores-100*(1-v**2))\n            result["newInformation_heuristic_check"] = {\n                "n_checked":int(len(check)),\n                "max_absolute_pct_point_difference":float(np.max(deviations)),\n                "median_absolute_pct_point_difference":float(np.median(deviations)),\n                "tested_formula":"100 * (1 - displayed_link_to_first**2)",\n                "not_expected_information_gain":True,\n            }\n    return result\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
rel='analysis/cfps_selection/input_sources.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Validated participant-pool input adapters for CFPS design selection.\n\nAccepted inputs:\n- ``published_snapshot``: the dated GP-informed ``clusters_designs.json`` export;\n- ``pocketbase_snapshot``: an authorized, read-only PocketBase snapshot containing\n  ``collections.cfps_designs`` or the corresponding paginated records payload.\n\nThe PocketBase adapter deliberately does not infer GP forecasts, cluster membership,\nrun roles, experimental output, or chemical stock concentrations. Consequently,\nGP-informed baseline and hybrid modes must not run on PocketBase-only input.\n\nThis module is embedded as an independent code cell in the Colab notebook.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nSOURCE_ADAPTER_VERSION = "1.1.0"\nSOURCE_TYPES = ("published_snapshot", "pocketbase_snapshot")\nCANONICAL_PROTEINS = ("sfGFP", "PETase", "Reteplase")\n\n\ndef _json_object(value: Any, label: str) -> dict:\n    if isinstance(value, str):\n        try:\n            value = json.loads(value)\n        except (ValueError, TypeError) as exc:\n            raise ValueError(f"{label} is not valid JSON") from exc\n    if not isinstance(value, Mapping):\n        raise ValueError(f"{label} must be a JSON object")\n    return dict(value)\n\n\ndef _canonical_target(value: Any) -> str:\n    text = str(value or "").strip().casefold().replace(" ", "").replace("-", "")\n    lookup = {"sfgfp": "sfGFP", "petase": "PETase", "reteplase": "Reteplase"}\n    if text not in lookup:\n        raise ValueError(f"Unknown/missing protein target: {value!r}; no target guessed from recipe")\n    return lookup[text]\n\n\ndef normalize_pocketbase_records(raw_snapshot: Mapping[str, Any], *,\n                                 require_complete: bool = True) -> tuple[dict, dict]:\n    """Create an analysis-compatible recipe pool from an authorized raw snapshot.\n\n    The output contains only fields verifiable in PocketBase; GP-derived fields are\n    *absent or null*. Invalid experimental records fail closed (no silent deletion).\n    Nonexperimental records are counted and excluded from the eligible pool.\n    """\n    if not isinstance(raw_snapshot, Mapping):\n        raise TypeError("PocketBase input must be an object")\n    meta = raw_snapshot.get("metadata") or {}\n    if meta and not isinstance(meta, Mapping):\n        raise ValueError("Snapshot metadata must be an object")\n    if require_complete and meta.get("designs_possibly_truncated") is True:\n        raise ValueError("PocketBase snapshot explicitly marked truncated; use a complete fetch")\n    if "collections" in raw_snapshot:\n        collections = _json_object(raw_snapshot["collections"], "collections")\n        records = collections.get("cfps_designs")\n    elif "cfps_designs" in raw_snapshot:\n        records = raw_snapshot["cfps_designs"]\n    elif "records" in raw_snapshot:\n        records = raw_snapshot["records"]\n    elif "items" in raw_snapshot:\n        records = raw_snapshot["items"]\n    else:\n        raise ValueError("No cfps_designs collection or records/items array")\n    if not isinstance(records, list):\n        raise ValueError("cfps_designs must be a list")\n    if require_complete and (meta.get("designs_advertised_total_items") is not None):\n        expected = int(meta["designs_advertised_total_items"])\n        if len(records) < expected:\n            raise ValueError(f"Incomplete PocketBase response: {len(records)}/{expected} records")\n    designs = {p: [] for p in CANONICAL_PROTEINS}\n    ids = set()\n    excluded_types = Counter()\n    missing_author_count = 0\n    for index, record in enumerate(records):\n        if not isinstance(record, Mapping):\n            raise ValueError(f"Record #{index} is not an object")\n        payload = _json_object(record.get("design_json"), f"record[{index}].design_json")\n        kind = str(payload.get("sample_type", "")).strip().lower()\n        if kind != "experimental":\n            excluded_types[kind or "missing"] += 1\n            continue\n        metadata = _json_object(payload.get("metadata") or {}, f"record[{index}].metadata")\n        target = _canonical_target(metadata.get("protein_target"))\n        recipe = _json_object(payload.get("reagent_list"), f"record[{index}].reagent_list")\n        if not recipe:\n            raise ValueError(f"Experimental record #{index} has no reagent_list")\n        converted = {}\n        total = 0.0\n        for k, value in recipe.items():\n            if isinstance(value, bool):\n                raise ValueError(f"Boolean volume {k} in record #{index}")\n            try:\n                volume = float(value)\n            except (TypeError, ValueError) as exc:\n                raise ValueError(f"Non-numeric reagent volume {k} in record #{index}") from exc\n            import math\n            if not math.isfinite(volume) or volume < 0:\n                raise ValueError(f"Invalid reagent volume {k} in record #{index}")\n            total += volume\n            if volume > 0:\n                converted[str(k)] = volume\n        if abs(total - 20000.0) > 1e-5:\n            raise ValueError(f"Experimental record #{index} total volume {total} nL, expected 20000 nL")\n        rid = str(record.get("id") or "").strip()\n        if not rid or rid in ids:\n            raise ValueError(f"Missing or duplicate immutable record ID: {rid!r}")\n        ids.add(rid)\n        author = (record.get("author") or record.get("author_sub") or\n                  record.get("author_name") or "")\n        if not author:\n            missing_author_count += 1\n        designs[target].append({\n            "id": rid,\n            "sampleId": str(payload.get("sample_id") or record.get("title") or rid),\n            "protein": target,\n            "created": record.get("created"),\n            "author": str(author),\n            "recipeNl": converted,\n            # Preserve explicit participant batch metadata for honest completeness checks.\n            # Never infer submitted batches from GP-defined clusters or sample ID prefixes.\n            "metadata": {key: metadata[key] for key in ("batch_id", "batch_size", "batch_index") if key in metadata},\n            # No GP-based properties fabricated; these are neutral placeholders.\n            "round1Role": None,\n            "sameRecipeAs": None,\n            "group": None,\n            "forecast": None,\n            "vsGinkgoPlates": None,\n            "dataOrigin": "PocketBase/cfps_designs",\n        })\n    if not ids:\n        raise ValueError("No valid experimental CFPS records in PocketBase snapshot")\n    out = {\n        "schema": "cfps-selection-input/v1",\n        "built": meta.get("created_utc") or None,\n        "input_source": "pocketbase_snapshot",\n        "model_based_annotations_available": False,\n        "proteins": {p: {"designs": designs[p], "groups": [], "sweeps": [],\n                         "totals": {"designs": len(designs[p])}}\n                     for p in CANONICAL_PROTEINS},\n    }\n    summary = {\n        "source_type": "pocketbase_snapshot",\n        "adapter_version": SOURCE_ADAPTER_VERSION,\n        "record_count": len(records),\n        "experimental_eligible": len(ids),\n        "counts_by_protein": {p: len(designs[p]) for p in CANONICAL_PROTEINS},\n        "nonexperimental_excluded": dict(sorted(excluded_types.items())),\n        "author_field_missing_count": missing_author_count,\n        "gp_baseline_available": False,\n        "quantity_semantics": "PocketBase reagent_list stored dispensing volumes (nL)",\n        "note": "No GP forecasts/roles were inferred; baseline and hybrid modes unavailable",\n    }\n    return out, summary\n\n\ndef normalize_analysis_source(path: str | Path, source_type: str) -> tuple[dict, dict]:\n    """Read an immutable source file and return compatible data with provenance.\n\n    The caller hashes both source bytes and any normalized cache file separately.\n    """\n    if source_type not in SOURCE_TYPES:\n        raise ValueError(f"source_type must be one of {SOURCE_TYPES}; got {source_type!r}")\n    source = Path(path)\n    raw = json.loads(source.read_text(encoding="utf-8"))\n    if source_type == "pocketbase_snapshot":\n        return normalize_pocketbase_records(raw)\n    if not isinstance(raw, dict) or not isinstance(raw.get("proteins"), dict):\n        raise ValueError("Published snapshot requires top-level proteins mapping")\n    for protein in CANONICAL_PROTEINS:\n        if protein not in raw["proteins"] or not isinstance(raw["proteins"][protein].get("designs"), list):\n            raise ValueError(f"Published snapshot is missing designs for {protein}")\n    return raw, {\n        "source_type": "published_snapshot", "adapter_version": SOURCE_ADAPTER_VERSION,\n        "counts_by_protein": {p: len(raw["proteins"][p]["designs"]) for p in CANONICAL_PROTEINS},\n        "gp_baseline_available": True,\n        "quantity_semantics": "published recipeNl dispensing volumes (nL)",\n        "note": "GP predictions and roles are source-provided, not regenerated",\n    }\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
rel='analysis/cfps_selection/decision_framework.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('"""Auditable, deterministic mixed-strategy CFPS plate planning.\n\nSubmitted formulations only; descriptive proxy objectives, NOT measured effects or\nvalidated GP predictive optima. First four columns of a 16x24 plate are reserved,\nper inspected Ginkgo CFPS example plate metadata. The 320 available physical wells\nare divided into independent formulations via a configurable replicate factor.\n\nNo secrets, network access, or private-data output paths are embedded here.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass, asdict\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Mapping\nimport hashlib\nimport json\nimport math\nimport numpy as np\nfrom scipy.spatial.distance import cdist\n\nfrom .selection_framework import canonical_recipe, input_sha256\n\nTARGETS = (\'sfGFP\', \'PETase\', \'Reteplase\')\nROLE_ORDER = {\'first run\': 0, \'check run\': 1, \'sweep run\': 2, \'independent run\': 3}\nRESERVED_COLUMNS = (1, 2, 3, 4)\nEXPERIMENTAL_WELLS = [f\'{chr(65+r)}{c:02d}\' for c in range(5,25) for r in range(16)]\nALL_WELLS = [f\'{chr(65+r)}{c:02d}\' for r in range(16) for c in range(1,25)]\n\n@dataclass(frozen=True)\nclass DecisionConfig:\n    replicate_factor: int = 4\n    gp_share: float = 0.50      # percentage normalized 0..1\n    gp_wells: int | None = None # override fraction; physical wells, multiple of replicate_factor\n    gp_policy: str = \'published_designations\'  # or \'forecast_ranked\'\n    model_free_objective: str = \'maximin\'        # or \'facility_location\' or \'balanced\'\n    geometry_weight: float = 0.5  # balanced mix maximin vs facility gain\n    presence_weight: float = 0.35\n    gp_yield_weight: float = 0.60\n    gp_cost_weight: float = 0.25\n    gp_uncertainty_weight: float = 0.15\n    source_mode: str = \'published_snapshot\'\n\n    def validate(self):\n        if self.replicate_factor <= 0 or 320 % self.replicate_factor:\n            raise ValueError(\'Replicate factor must be a positive divisor of 320\')\n        if not 0 <= self.gp_share <= 1:\n            raise ValueError(\'gp_share must lie in [0, 1]\')\n        if self.gp_wells is not None and (not isinstance(self.gp_wells, int) or\n                 self.gp_wells < 0 or self.gp_wells > 320 or self.gp_wells % self.replicate_factor):\n            raise ValueError(\'gp_wells must be 0..320 and divisible by replicate_factor\')\n        if self.gp_policy not in {\'published_designations\',\'forecast_ranked\'}:\n            raise ValueError(\'Unknown GP policy\')\n        if self.model_free_objective not in {\'maximin\',\'facility_location\',\'balanced\'}:\n            raise ValueError(\'Unknown geometry objective\')\n        for v in (self.geometry_weight,self.presence_weight,self.gp_yield_weight,self.gp_cost_weight,self.gp_uncertainty_weight):\n            if not math.isfinite(v) or v < 0:raise ValueError(\'Weights must be nonnegative and finite\')\n        if self.geometry_weight > 1:raise ValueError(\'geometry_weight must be <=1\')\n        if self.source_mode not in {\'published_snapshot\',\'pocketbase_snapshot\',\'pocketbase_live\'}:\n            raise ValueError(\'Unknown source mode\')\n        return self\n\n    @property\n    def unique_capacity(self):\n        return 320//self.replicate_factor\n\n    @property\n    def gp_unique_quota(self):\n        return (self.gp_wells//self.replicate_factor if self.gp_wells is not None\n                else math.floor(self.unique_capacity * self.gp_share + 0.5))\n\n\ndef batch_id_for_design(d):\n    """Only explicitly declared batch IDs qualify as submission batches."""\n    meta = d.get(\'metadata\') if isinstance(d.get(\'metadata\'),dict) else {}\n    value = d.get(\'batch_id\') or meta.get(\'batch_id\')\n    return str(value) if value else None\n\n\ndef candidate_payload(data: Mapping[str,Any], target:str, *, presence_weight:float=.35) -> dict:\n    """Compact authenticated web input from a frozen GP-annotated export or normalized PB data.\n\n    Includes participant-level identifiers: NEVER write to static/ or Git.\n    """\n    if target not in TARGETS:raise ValueError(\'Unknown protein\')\n    block = data[\'proteins\'][target]\n    groups=block.get(\'groups\',[])\n    sweeps=block.get(\'sweeps\',[])\n    raw=block[\'designs\']\n    canonical=defaultdict(list)\n    for d in raw:\n        canonical[canonical_recipe(d[\'recipeNl\'])].append(d)\n    reps=[]\n    for _,dupes in canonical.items():\n        dupes.sort(key=lambda d:(ROLE_ORDER.get(d.get(\'round1Role\'),99),str(d[\'id\'])))\n        representative=dupes[0]\n        rep=dict(representative)\n        rep[\'aliases\']=sorted(str(d[\'id\']) for d in dupes)\n        rep[\'batch_ids\']=sorted({x for d in dupes if (x:=batch_id_for_design(d))})\n        rep[\'roles\']=sorted({d[\'round1Role\'] for d in dupes if d.get(\'round1Role\') in ROLE_ORDER}, key=lambda r:ROLE_ORDER[r])\n        reps.append(rep)\n    reps.sort(key=lambda d:str(d[\'id\']))\n    factors=sorted({k for r in reps for k,v in r[\'recipeNl\'].items() if k!=\'nuclease_free_water\' and float(v)!=0})\n    raw_matrix=np.array([[float(r[\'recipeNl\'].get(k,0)) for k in factors] for r in reps],dtype=float)\n    positive=np.array([np.quantile(raw_matrix[raw_matrix[:,i]>0,i],.95) if np.any(raw_matrix[:,i]>0) else 1. for i in range(len(factors))])\n    scaled=np.minimum(1,raw_matrix/np.maximum(1e-9,positive))\n    presence=(raw_matrix>0).astype(float)\n    features=np.hstack([scaled, math.sqrt(presence_weight)*presence])\n    fmax=cdist(features,features).max() if len(reps)>1 else 1.\n    if fmax==0:fmax=1\n    # Published groups, sweeps and batch IDs are stored separately: they are not interchangeable.\n    id_alias={alias:r[\'id\'] for r in reps for alias in r[\'aliases\']}\n    published_groups=[]\n    for g in groups:\n        ids=sorted({id_alias[i] for i in g.get(\'designs\',[]) if i in id_alias})\n        published_groups.append({\'name\':str(g[\'letter\']),\'ids\':ids,\'firstRun\':id_alias.get(g.get(\'firstRun\')),\n                                 \'checkRuns\':[id_alias.get(c[\'id\']) for c in g.get(\'checkRuns\',[]) if c.get(\'id\') in id_alias]})\n    sweep_families=[]\n    for group in sweeps:\n        for idx,s in enumerate(group.get(\'sweeps\',[])):\n            ids=sorted({id_alias[x.get(\'id\')] for x in s.get(\'toRun\',[]) if isinstance(x,dict) and x.get(\'id\') in id_alias})\n            if len(ids)>1:sweep_families.append({\'name\':f\'{group["reagent"]} #{idx+1}\',\'ids\':ids})\n    explicit=defaultdict(list)\n    expected_batch_sizes=defaultdict(set)\n    for d in raw:\n        batch=batch_id_for_design(d)\n        meta=d.get(\'metadata\') if isinstance(d.get(\'metadata\'),dict) else {}\n        declared=meta.get(\'batch_size\') or d.get(\'batch_size\')\n        if batch and declared is not None:\n            try: value=int(declared)\n            except (TypeError,ValueError):value=0\n            if value>0:expected_batch_sizes[batch].add(value)\n    for r in reps:\n        for batch in r[\'batch_ids\']:explicit[batch].append(r[\'id\'])\n    has_gp=bool(groups) and any(r.get(\'forecast\') for r in reps)\n    return {\'target\':target,\'source_built\':data.get(\'built\'),\'n_original_submissions\':len(raw),\n            \'n_distinct\':len(reps),\'source_has_gp_annotations\':has_gp,\n            \'metric\':{\'name\':\'p95-scaled dispensing-volume Euclidean distance with reagent presence\',\n                      \'presence_weight\':presence_weight,\'normalizer\':float(fmax),\'factor_keys\':factors,\n                      \'scale_p95_nL\':positive.tolist(), \'not_molarity\':True},\n            \'candidates\':[{\'id\':r[\'id\'],\'aliases\':r[\'aliases\'],\'sampleId\':r.get(\'sampleId\'),\'author\':r.get(\'author\'),\n                    \'recipeNl\':r[\'recipeNl\'],\'features\':[round(float(x),7) for x in features[i]],\n                    \'forecast\':r.get(\'forecast\'),\'role\':r.get(\'round1Role\'),\'roles\':r[\'roles\'],\n                    \'groups\':sorted({str(d[\'group\'][\'letter\']) for d in raw if d[\'id\'] in r[\'aliases\'] and d.get(\'group\')}),\n                    \'batch_ids\':r[\'batch_ids\']} for i,r in enumerate(reps)],\n            \'published_groups\':published_groups,\'sweep_families\':sweep_families,\n            \'submission_batches\':[{\'name\':b,\'ids\':sorted(set(ids)),\n                        \'reported_size\':next(iter(expected_batch_sizes[b])) if len(expected_batch_sizes[b])==1 else None,\n                        \'reported_sizes_disagree\':len(expected_batch_sizes[b])>1,\n                        \'possible_missing_source_members\':(max(0,next(iter(expected_batch_sizes[b]))-len(ids)) if len(expected_batch_sizes[b])==1 else None)}\n                        for b,ids in sorted(explicit.items()) if len(set(ids))>1 or expected_batch_sizes[b]]}\n\n\ndef _gp_scores(candidates,cfg):\n    fields=[(\'pBeat\',cfg.gp_yield_weight,1),(\'lnSpecCost\',cfg.gp_cost_weight,-1),(\'sd\',cfg.gp_uncertainty_weight,1)]\n    score=np.zeros(len(candidates),float)\n    denom=sum(w for _,w,_ in fields)\n    if denom==0:return score\n    for field,w,sign in fields:\n        vals=np.array([(c.get(\'forecast\') or {}).get(field) for c in candidates],dtype=object)\n        nums=np.array([float(v) if v is not None and math.isfinite(float(v)) else np.nan for v in vals])\n        valid=np.isfinite(nums)\n        if valid.sum()<2 or np.nanmax(nums)==np.nanmin(nums):continue\n        order=np.argsort(nums[valid],kind=\'stable\')\n        ranks=np.empty(order.size)\n        ranks[order]=np.arange(order.size)/(order.size-1)\n        part=np.full(len(candidates),.0)\n        part[valid]=ranks if sign>0 else (1-ranks)\n        score+=w/denom*part\n    return score\n\n\ndef _distance_matrix(candidates):\n    X=np.array([c[\'features\'] for c in candidates],float)\n    d=cdist(X,X)\n    return d / max(float(d.max()),1e-12)\n\n\ndef detect_partial_sets(payload,selected):\n    """Explicit submission batches are authoritative; GP groups/sweeps advisory."""\n    selected=set(selected);warnings=[]\n    for typ,field,severity in [(\'submitted_batch\',\'submission_batches\',\'HIGH\'),\n                               (\'published_gp_group\',\'published_groups\',\'INFO\'),\n                               (\'model_sweep\',\'sweep_families\',\'WARNING\')]:\n        for g in payload.get(field,[]):\n            ids=set(g[\'ids\'])\n            present=sorted(ids & selected)\n            missing=sorted(ids - selected)\n            if present and missing:\n                warnings.append({\'type\':typ,\'severity\':severity,\'name\':g[\'name\'],\n                                 \'selected_count\':len(present),\'expected_count\':len(ids),\n                                 \'selected_ids\':present,\'missing_ids\':missing,\n                                 \'all_or_none_required\':typ==\'submitted_batch\'})\n    for group in payload.get(\'submission_batches\',[]):\n        if not (set(group.get(\'ids\',[])) & selected):continue\n        if group.get(\'reported_sizes_disagree\') or (group.get(\'possible_missing_source_members\') or 0)>0:\n            warnings.append({\'type\':\'source_batch_incomplete\',\'severity\':\'HIGH\',\'name\':group[\'name\'],\n                \'selected_count\':len(set(group[\'ids\']) & selected),\'expected_count\':group.get(\'reported_size\'),\n                \'selected_ids\':sorted(set(group[\'ids\']) & selected),\'missing_ids\':[],\n                \'missing_source_count\':group.get(\'possible_missing_source_members\'),\n                \'all_or_none_required\':True})\n    return warnings\n\n\ndef select_plate(payload:dict, cfg:DecisionConfig=DecisionConfig(),\n                 *, locked_gp=(),locked_model_free=(),excluded=()):\n    """Exact GP/MF quota with locked manual choices. No implicit fallback across methods.\n\n    Published GP roles are used *as published*. A GP quota beyond those roles\n    is extended by forecast ranking and recorded explicitly. Unannotated PB data\n    is rejected for GP share > 0. Returns a suggestion, never execution-ready.\n    """\n    cfg.validate(); candidates=payload[\'candidates\'];n=len(candidates)\n    capacity=cfg.unique_capacity\n    if n<capacity:raise ValueError(f\'Only {n} distinct recipes for capacity {capacity}\')\n    gp_capacity=cfg.gp_unique_quota;mf_capacity=capacity-gp_capacity\n    if gp_capacity and not payload.get(\'source_has_gp_annotations\'):raise ValueError(\'GP quota requires annotated published snapshot\')\n    ids={c[\'id\']:i for i,c in enumerate(candidates)}\n    locked_gp=set(locked_gp);locked_model_free=set(locked_model_free);excluded=set(excluded)\n    invalid=(locked_gp | locked_model_free | excluded)-ids.keys()\n    if invalid:raise ValueError(f\'Unknown recipe ID(s): {sorted(invalid)[:5]}\')\n    if (locked_gp & locked_model_free) or ((locked_gp|locked_model_free)&excluded):raise ValueError(\'Conflicting manual locks\')\n    if len(locked_gp)>gp_capacity or len(locked_model_free)>mf_capacity:raise ValueError(\'Manual locks exceed method quota\')\n    if n-len(excluded)<capacity:raise ValueError(\'Too few eligible recipes\')\n    gp_scores=_gp_scores(candidates,cfg)\n    picked_gp=list(sorted(locked_gp))\n    chosen=set(locked_gp)|set(locked_model_free)\n    gp_extension=[]\n    available=[i for i,c in enumerate(candidates) if c[\'id\'] not in chosen|excluded]\n    if cfg.gp_policy==\'published_designations\':\n        available.sort(key=lambda i:(min((ROLE_ORDER.get(role,99) for role in candidates[i].get(\'roles\',[])),default=99),\n                                      -gp_scores[i],candidates[i][\'id\']))\n    else:available.sort(key=lambda i:(-gp_scores[i],candidates[i][\'id\']))\n    for i in available:\n        if len(picked_gp)>=gp_capacity:break\n        c=candidates[i]\n        picked_gp.append(c[\'id\']);chosen.add(c[\'id\'])\n        if not c.get(\'roles\'):gp_extension.append(c[\'id\'])\n    if len(picked_gp)!=gp_capacity:raise ValueError(\'Unable to satisfy GP quota\')\n    # Greedy normalized chemical-distance coverage. Seeded by GP+manual selections.\n    D=_distance_matrix(candidates)\n    idx=np.array([ids[i] for i in chosen],int)\n    nearest=np.min(D[:,idx],axis=1) if len(idx) else np.full(n,1.,float)\n    picked_free=sorted(locked_model_free)\n    while len(picked_free)<mf_capacity:\n        eligible=[i for i,c in enumerate(candidates) if c[\'id\'] not in chosen|excluded]\n        if not eligible:raise ValueError(\'Insufficient distinct eligible candidate recipes\')\n        if cfg.model_free_objective==\'maximin\':scores=nearest[eligible]\n        else:\n            facilities=np.maximum(0.,nearest[None,:]-D[eligible,:]).mean(axis=1)\n            if cfg.model_free_objective==\'facility_location\':scores=facilities\n            else:scores=cfg.geometry_weight*nearest[eligible]+(1-cfg.geometry_weight)*facilities\n        best=eligible[min(range(len(eligible)),key=lambda j:(-float(scores[j]),candidates[eligible[j]][\'id\']))]\n        rid=candidates[best][\'id\'];picked_free.append(rid);chosen.add(rid)\n        nearest=np.minimum(nearest,D[:,best])\n    # Column-major physical order (as in supplied 384-well plate examples).\n    assignment=[];ordered=[(\'gp\',i) for i in picked_gp]+[(\'model_free\',i) for i in picked_free]\n    physical=[x for category,rid in ordered for x in range(cfg.replicate_factor)]\n    if len(physical)!=320:raise AssertionError(\'320 available wells must be mapped\')\n    for i,(method,rid) in enumerate(ordered):\n        for replicate in range(cfg.replicate_factor):\n            assignment.append({\'well\':EXPERIMENTAL_WELLS[i*cfg.replicate_factor+replicate],\n                               \'id\':rid,\'method\':method,\'replicate\':replicate+1})\n    warnings=detect_partial_sets(payload,chosen)\n    return {\'schema\':\'cfps-decision-plan/v1\',\'target\':payload[\'target\'],\'status\':\'PROVISIONAL, requires verification of plate rules, batches, physical layout and approval\',\n            \'config\':asdict(cfg),\'unique_capacity\':capacity,\'gp_unique_quota\':gp_capacity,\n            \'model_free_unique_quota\':mf_capacity,\'gp_well_quota\':gp_capacity*cfg.replicate_factor,\n            \'model_free_well_quota\':mf_capacity*cfg.replicate_factor,\n            \'selected_ids\':sorted(chosen),\'by_method\':{\'gp\':picked_gp,\'model_free\':picked_free},\n            \'gp_forecast_extensions\':gp_extension,\n            \'reserved_columns\':list(RESERVED_COLUMNS),\'experimental_wells\':EXPERIMENTAL_WELLS,\n            \'well_assignment\':assignment,\'warnings\':warnings,\n            \'partial_submitted_batches\':sum(w[\'type\'] in (\'submitted_batch\',\'source_batch_incomplete\') for w in warnings),\n            \'coverage_mean_nearest_distance\':float(np.mean(nearest)),\n            \'distance_not_empirical_biology\':True}\n\n\ndef verify_published_groups(data:Mapping[str,Any]) -> dict:\n    """Internal GP export integrity + explicit non-equivalence of independent clustering.\n\n    Independent raw-volume Pearson clusters are NOT reconstructing GP shared-error\n    links, hence equality cannot be asserted without the original GP algorithm.\n    """\n    out={\'source\':\'original GP-annotated clusters_designs.json\',\n         \'independent_cluster_equivalence\':\'NOT ESTABLISHED — algorithms represent different quantities\',\n         \'published_designations_directly_available\':True,\'proteins\':{}}\n    for target in TARGETS:\n        block=data[\'proteins\'][target];all_ids={d[\'id\'] for d in block[\'designs\']}\n        errs=[];count=0\n        for g in block[\'groups\']:\n            members=set(g[\'designs\']);count+=len(members)\n            if len(members)!=len(g[\'designs\']):errs.append(f\'duplicate IDs group {g["letter"]}\')\n            if members-all_ids:errs.append(f\'unknown design IDs group {g["letter"]}\')\n            if g.get(\'firstRun\') not in members:errs.append(f\'first run absent from {g["letter"]}\')\n            for c in g.get(\'checkRuns\',[]):\n                if c[\'id\'] not in members:errs.append(f\'check run absent from {g["letter"]}\')\n        source_group={rid:g[\'letter\'] for g in block[\'groups\'] for rid in g[\'designs\']}\n        for design in block[\'designs\']:\n            annotated=(design.get(\'group\') or {}).get(\'letter\')\n            if annotated != source_group.get(design[\'id\']):\n                errs.append(f\'group membership annotation disagreement for {design["id"]}\')\n        out[\'proteins\'][target]={\'n_published_groups\':len(block[\'groups\']),\n                                 \'n_group_memberships\':count,\'errors\':errs,\'integrity_ok\':not errs}\n    return out\n\n\ndef write_private_input(data:Mapping[str,Any],path:str|Path,*,provenance:dict|None=None):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    obj={\'schema\':\'cfps-decision-input/v1\',\'privacy\':\'PRIVATE; DO NOT COMMIT TO GIT OR STATIC\',\n         \'provenance\':provenance or {},\'layout\':{\'n_rows\':16,\'n_columns\':24,\'reserved_columns\':[1,2,3,4],\n                    \'experimental_columns\':list(range(5,25)),\'array_order\':\'column\',\n                    \'source\':\'inspected cfps-plates.zip example plate metadata\'},\n         \'cluster_parity\':verify_published_groups(data) if all(data.get(\'proteins\',{}).get(t,{}).get(\'groups\') for t in TARGETS) else {\'independent_cluster_equivalence\':\'NOT ESTABLISHED\'},\n         \'targets\':{t:candidate_payload(data,t) for t in TARGETS}}\n    path.write_text(json.dumps(obj,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False),encoding=\'utf8\')\n    return {\'path\':str(path),\'sha256\':input_sha256(path),\'bytes\':path.stat().st_size}\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
rel='tools/cfps_decision_cli.py'
path=ROOT/rel
path.parent.mkdir(parents=True,exist_ok=True)
path.write_text('#!/usr/bin/env python3\n"""Private-input, read-only decision planning CLI (never a robot-ready plan)."""\nfrom __future__ import annotations\nimport argparse\nfrom dataclasses import replace\nfrom pathlib import Path\nimport json\nimport sys\nROOT=Path(__file__).resolve().parents[1]\nsys.path.insert(0,str(ROOT))\nfrom analysis.cfps_selection.decision_framework import DecisionConfig,TARGETS,write_private_input,select_plate,candidate_payload,verify_published_groups\nfrom analysis.cfps_selection.input_sources import normalize_analysis_source\nfrom analysis.cfps_selection.selection_framework import input_sha256\n\n\ndef run(source_path:str, out_dir:str, *, source_type:str=\'published_snapshot\',\n        replicate_factor:int=4, gp_share:float=.5, gp_wells:int|None=None,\n        gp_policy:str=\'published_designations\', objective:str=\'maximin\')->dict:\n    cfg=DecisionConfig(replicate_factor=replicate_factor, gp_share=gp_share,\n                       gp_wells=gp_wells,gp_policy=gp_policy,\n                       model_free_objective=objective,\n                       source_mode=source_type)\n    cfg.validate()\n    data,summary=normalize_analysis_source(source_path,source_type)\n    dest=Path(out_dir);dest.mkdir(parents=True,exist_ok=True)\n    (dest/\'.gitignore\').write_text(\'*\\n!.gitignore\\n\')\n    provenance={\'input_source_type\':source_type,\'source_sha256\':input_sha256(source_path),\n                \'normalization\':summary,\'warning\':\'PRIVATE — never commit participant-level output\'}\n    generated=write_private_input(data,dest/\'cfps_decision_input_private.json\',provenance=provenance)\n    result={\'schema\':\'cfps-decision-plan-collection/v1\',\'privacy\':\'PRIVATE DO NOT COMMIT\',\n            \'provenance\':provenance,\'input\':generated,\n            \'cluster_parity\':verify_published_groups(data) if source_type==\'published_snapshot\' else None,\n            \'plans\':{}}\n    for target in TARGETS:\n        payload=candidate_payload(data,target,presence_weight=cfg.presence_weight)\n        result[\'plans\'][target]=select_plate(payload,cfg)\n    (dest/\'decision_plans_private.json\').write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False))\n    warnings=[dict(target=t,**w) for t,p in result[\'plans\'].items() for w in p[\'warnings\']]\n    (dest/\'decision_warnings_private.json\').write_text(json.dumps(warnings,ensure_ascii=False,indent=2,allow_nan=False))\n    with (dest/\'decision_well_assignments_private.csv\').open(\'w\') as f:\n        f.write(\'protein,well,submission_id,selection_method,replicate,provisional\\n\')\n        for t,p in result[\'plans\'].items():\n            for a in p[\'well_assignment\']:\n                f.write(f\'{t},{a["well"]},{a["id"]},{a["method"]},{a["replicate"]},true\\n\')\n    print(json.dumps({\'input\':str(generated[\'path\']), \'unique_conditions_per_plate\':cfg.unique_capacity,\n                      \'GP_conditions_per_plate\':cfg.gp_unique_quota,\n                      \'warnings\':{t:len(result[\'plans\'][t][\'warnings\']) for t in TARGETS},\n                      \'privacy\':\'PRIVATE; not committed\'},indent=2))\n    return result\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--source\',choices=[\'published_snapshot\',\'pocketbase_snapshot\'],default=\'published_snapshot\')\n    p.add_argument(\'--input\',required=True)\n    p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--replicates\',type=int,default=4)\n    p.add_argument(\'--gp-share\',type=float,default=.5)\n    p.add_argument(\'--gp-wells\',type=int)\n    p.add_argument(\'--gp-policy\',choices=[\'published_designations\',\'forecast_ranked\'],default=\'published_designations\')\n    p.add_argument(\'--objective\',choices=[\'maximin\',\'facility_location\',\'balanced\'],default=\'maximin\')\n    a=p.parse_args()\n    run(a.input,a.output,source_type=a.source,replicate_factor=a.replicates,gp_share=a.gp_share,\n        gp_wells=a.gp_wells,gp_policy=a.gp_policy,objective=a.objective)\n',encoding='utf-8')
print('Embedded module:',rel,'bytes:',path.stat().st_size)

In [ ]:
import sys
sys.path.insert(0,str(ROOT))
from analysis.cfps_selection.input_sources import normalize_analysis_source
from analysis.cfps_selection.decision_framework import (
    DecisionConfig,TARGETS,candidate_payload,select_plate,
    verify_published_groups,write_private_input)
from analysis.cfps_selection.selection_framework import input_sha256


## 3 · Acquire an authorized, immutable snapshot

For live PocketBase, provide a token through Colab Secrets (`PB_TOKEN`) **or** authenticate interactively using email/password. This cell never prints credentials or writes tokens to disk. `PB_TOKEN` is not a Keycloak token; do not use a Keycloak confidential-client secret for PocketBase authentication.

In [ ]:
from google.colab import files
if INPUT_SOURCE not in {'published_snapshot','pocketbase_snapshot','pocketbase_live'}:
    raise ValueError('Unsupported source mode')
if INPUT_SOURCE == 'pocketbase_live':
    import requests
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    def private_value(name):
        x=os.environ.get(name,'')
        if not x and userdata is not None:
            try: x=userdata.get(name) or ''
            except Exception: pass
        return x
    token=private_value('PB_TOKEN')
    if not token:
        email=private_value('PB_EMAIL') or input('PocketBase authorized email: ')
        pw=private_value('PB_PASSWORD') or getpass.getpass('PocketBase password: ')
        reply=requests.post(f'{PB_URL.rstrip("/")}/api/collections/{PB_AUTH_COLLECTION}/auth-with-password',
                 json={'identity':email,'password':pw},timeout=30)
        reply.raise_for_status()
        token=reply.json()['token']
        del pw
    http=requests.Session()
    http.headers.update({'Authorization':token,'Accept':'application/json'})
    records=[];page=1;expected=None
    while True:
        reply=http.get(f'{PB_URL.rstrip("/")}/api/collections/cfps_designs/records',
                       params={'page':page,'perPage':200,'sort':'id'},timeout=60)
        reply.raise_for_status()
        body=reply.json()
        if expected is None: expected=int(body['totalItems'])
        batch=body['items']; records.extend(batch)
        if not batch or len(records)>=expected:break
        page+=1
    assert len(records)==expected, 'Incomplete paginated fetch: FAIL CLOSED'
    assert len({str(r['id']) for r in records})==len(records),'Duplicate immutable record IDs'
    source_path=PRIVATE/'pocketbase_snapshot_private.json'
    source_path.write_text(json.dumps({'metadata':{'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'designs_advertised_total_items':expected,'designs_possibly_truncated':False,'source':'PocketBase/cfps_designs'},
      'collections':{'cfps_designs':records}},ensure_ascii=False),encoding='utf8')
    del token
    source_type='pocketbase_snapshot'
else:
    source_type=INPUT_SOURCE
    source_path=Path(INPUT_PATH)
    if not source_path.exists():
        print('Upload your authorized, private source JSON file:')
        up=files.upload()
        if not up:raise ValueError('No source uploaded')
        source_path=PRIVATE/next(iter(up))
        source_path.write_bytes(next(iter(up.values())))
print('Input mode:',INPUT_SOURCE,'input SHA-256:',input_sha256(source_path))


## 4 · Validate input, original groups, quotas, and source provenance

In [ ]:
data,adapter_summary=normalize_analysis_source(source_path,source_type)
config=DecisionConfig(replicate_factor=REPLICATE_FACTOR,
    gp_share=GP_SHARE_PERCENT/100,gp_wells=GP_WELLS,gp_policy=GP_POLICY,
    model_free_objective=MODEL_FREE_CRITERION,geometry_weight=BALANCED_MAXIMIN_WEIGHT,
    gp_yield_weight=GP_YIELD_WEIGHT,gp_cost_weight=GP_COST_WEIGHT,
    gp_uncertainty_weight=GP_UNCERTAINTY_WEIGHT,source_mode=INPUT_SOURCE).validate()
print('Input normalization:',json.dumps(adapter_summary,indent=2))
if INPUT_SOURCE=='published_snapshot':
    integrity=verify_published_groups(data)
    print('Published group consistency:',json.dumps(integrity,indent=2))
    assert all(s['integrity_ok'] for s in integrity['proteins'].values()),'Published group references inconsistent'
else:
    integrity={'independent_cluster_equivalence':'GP groups are absent; no equivalence asserted'}
    assert config.gp_unique_quota==0, 'PocketBase-only mode CANNOT allocate GP quota'
print('Capacity:',config.unique_capacity,'unique formulations per plate;',config.replicate_factor,'replicates per formulation')
print('Requested allocations: GP',config.gp_unique_quota,'model-free',config.unique_capacity-config.gp_unique_quota)


## 5 · Produce provisional decisions and all 384-well visualizations

In [ ]:
from dataclasses import asdict
provenance={'source_mode':INPUT_SOURCE,'source_sha256':input_sha256(source_path),
            'adapter_summary':adapter_summary,'published_group_integrity':integrity,
            'warning':'Participant-level private input; do not publish'}
source_result=write_private_input(data,PRIVATE/'cfps_decision_input_private.json',provenance=provenance)
plans={t:select_plate(candidate_payload(data,t,presence_weight=config.presence_weight),config) for t in TARGETS}
output={'schema':'cfps-decision-plan-collection/v1','privacy':'PRIVATE; DO NOT COMMIT',
        'provenance':provenance,'plans':plans}
(PRIVATE/'decision_plans_private.json').write_text(json.dumps(output,ensure_ascii=False,indent=2,allow_nan=False))
rows=['protein,well,submission_id,method,replicate,provisional']
for target,plan in plans.items():
    print(target,'GP:',plan['gp_unique_quota'],'Model-free:',plan['model_free_unique_quota'],
          'GP forecast-only extensions:',len(plan['gp_forecast_extensions']),
          'Partial explicit batches:',plan['partial_submitted_batches'],
          'All partial analytical families:',len(plan['warnings']))
    for a in plan['well_assignment']:
        rows.append(','.join(map(str,[target,a['well'],a['id'],a['method'],a['replicate'],'true'])))
(PRIVATE/'decision_well_assignments_private.csv').write_text('\n'.join(rows)+'\n')
(PRIVATE/'decision_warnings_private.json').write_text(json.dumps({t:p['warnings'] for t,p in plans.items()},indent=2,ensure_ascii=False))


## 6 · Offline interactive dashboard

The HTML dashboard is generated with the *same* JavaScript and CSS as the authenticated SvelteKit route. Load `cfps_decision_input_private.json` using its on-page file picker, or use the embedded preload version below (contains private submissions; never commit it). The dashboard allows manual well assignments and download of edited plans.

In [ ]:
viewer_js='/* HTGAA CFPS Decision Lab: private, client-side, deterministic selection and physical plate map.\n   No network writes, credentials, cross-origin analytics or third-party dependencies.\n   This is a suggested plan, never a validated robot execution file. */\n(function(root) {\n\'use strict\';\nconst PROTEINS=[\'sfGFP\',\'PETase\',\'Reteplase\'], ROLE={\'first run\':0,\'check run\':1,\'sweep run\':2,\'independent run\':3};\nconst ROWS=\'ABCDEFGHIJKLMNOP\'.split(\'\'), expWells=Array.from({length:20},(_,c)=>c+5).flatMap(c=>ROWS.map(r=>r+String(c).padStart(2,\'0\')));\nconst esc=x=>String(x==null?\'\':x).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst num=(x, fallback=0)=>{const n=Number(x);return Number.isFinite(n)?n:fallback};\nconst chooseCount=(cfg)=>cfg.quotaMode===\'wells\'?(cfg.gpWells%cfg.reps?NaN:cfg.gpWells/cfg.reps):Math.floor((320/cfg.reps)*cfg.gpPercent/100+.5);\nconst uniqCapacity=cfg=>320/cfg.reps;\nfunction distanceMatrix(cands){const n=cands.length, data=new Float32Array(n*n);let max=0;for(let i=0;i<n;i++){for(let j=0;j<i;j++){const a=cands[i].features,b=cands[j].features;let s=0;for(let k=0;k<a.length;k++){const z=a[k]-b[k];s+=z*z}const d=Math.sqrt(s);data[i*n+j]=data[j*n+i]=d;if(d>max)max=d}}for(let i=0;i<data.length;i++)data[i]/=max||1;return data}\nfunction gpScores(cands,cfg){const fields=[[\'pBeat\',cfg.gpYield,1],[\'lnSpecCost\',cfg.gpCost,-1],[\'sd\',cfg.gpSD,1]],denom=fields.reduce((s,x)=>s+x[1],0);const scores=new Float64Array(cands.length);if(!denom)return scores;\n  for(const [field,w,sign] of fields){if(!w)continue;const vals=cands.map((c,i)=>({i,v:Number(c.forecast?.[field])})).filter(o=>cands[o.i].forecast?.[field]!=null&&Number.isFinite(o.v)).sort((a,b)=>a.v-b.v||a.i-b.i);if(vals.length<2||vals[0].v===vals[vals.length-1].v)continue;vals.forEach((o,r)=>scores[o.i]+=(w/denom)*(sign>0?r/(vals.length-1):1-r/(vals.length-1)));}return scores}\nfunction partialSets(p,selected){const warnings=[];for(const [kind,key,severity] of [[\'submitted_batch\',\'submission_batches\',\'high\'],[\'published_gp_group\',\'published_groups\',\'info\'],[\'model_sweep\',\'sweep_families\',\'medium\']])for(const g of p[key]||[]){const inS=g.ids.filter(x=>selected.has(x));const miss=g.ids.filter(x=>!selected.has(x));if(inS.length&&miss.length)warnings.push({kind,severity,name:g.name,nSelected:inS.length,total:g.ids.length,missing:miss,mandatory:kind===\'submitted_batch\'})}for(const g of p.submission_batches||[]){if(!g.ids.some(x=>selected.has(x)))continue;if(g.reported_sizes_disagree||g.possible_missing_source_members>0)warnings.push({kind:\'source_batch_incomplete\',severity:\'high\',name:g.name,nSelected:g.ids.filter(x=>selected.has(x)).length,total:g.reported_size||\'? \',missing:[],mandatory:true,sourceMissing:g.possible_missing_source_members||0})}return warnings}\nfunction solve(input,target,cfg,manual,cache){const p=input.targets[target],cands=p.candidates,n=cands.length,capacity=uniqCapacity(cfg),gq=chooseCount(cfg),mq=capacity-gq;\n if(!Number.isInteger(capacity)||!Number.isInteger(gq)||gq<0||gq>capacity)throw Error(\'Invalid quota: GP wells must be divisible by replicate factor\');\n if(capacity>n)throw Error(`Insufficient distinct submitted formulations (${n} available for ${capacity} conditions)`);\n if(gq&&!p.source_has_gp_annotations)throw Error(\'This source lacks published GP annotations. Set GP share to zero.\');\n const idx=Object.fromEntries(cands.map((c,i)=>[c.id,i]));const overrides=manual[target]?.slots||{},excluded=new Set(manual[target]?.excluded||[]);\n const fixedGP=[],fixedMF=[],fixedSlots=[];\n for(const [key,id] of Object.entries(overrides)){const slot=Number(key);if(!Number.isInteger(slot)||slot<0||slot>=capacity||idx[id]===undefined)continue;const m=slot<gq?\'gp\':\'model_free\';(m===\'gp\'?fixedGP:fixedMF).push(id);fixedSlots.push({slot,id,method:m})}\n if(new Set(fixedSlots.map(x=>x.id)).size!==fixedSlots.length)throw Error(\'One manually chosen submission is assigned to multiple slots.\');\n if(fixedGP.length>gq||fixedMF.length>mq)throw Error(\'Manual assignments exceed the quota.\');\n for(const id of fixedGP.concat(fixedMF))excluded.delete(id);\n if(n-excluded.size<capacity)throw Error(\'Too many exclusions for this plate capacity\');\n const selected=new Set([...fixedGP,...fixedMF]);const gp=fixedGP.slice(),mf=fixedMF.slice(),extensions=[];\n const scores=gpScores(cands,cfg);const avail=cands.map((c,i)=>i).filter(i=>!selected.has(cands[i].id)&&!excluded.has(cands[i].id));\n const rank=i=>Math.min(99,...(cands[i].roles||[]).map(x=>ROLE[x]??99));\n avail.sort((a,b)=> (cfg.gpPolicy===\'published_designations\'?rank(a)-rank(b):0)||scores[b]-scores[a]||cands[a].id.localeCompare(cands[b].id));\n for(const i of avail){if(gp.length>=gq)break;let id=cands[i].id;gp.push(id);selected.add(id);if(!(cands[i].roles||[]).length)extensions.push(id)}\n if(gp.length!==gq)throw Error(\'GP quota cannot be filled\');\n const D=cache[target]||(cache[target]=distanceMatrix(cands)),nearest=new Float64Array(n).fill(1);\n for(const id of selected){const i=idx[id];for(let j=0;j<n;j++)nearest[j]=Math.min(nearest[j],D[i*n+j])}\n while(mf.length<mq){let best=-1,bestScore=-Infinity;\n   for(let i=0;i<n;i++){const id=cands[i].id;if(selected.has(id)||excluded.has(id))continue;let value=nearest[i];if(cfg.objective!==\'maximin\'){let gain=0;for(let j=0;j<n;j++)gain+=Math.max(0,nearest[j]-D[i*n+j]);gain/=n;value=cfg.objective===\'facility_location\'?gain:cfg.geoWeight*value+(1-cfg.geoWeight)*gain}\n     if(value>bestScore+1e-10||(Math.abs(value-bestScore)<1e-10&&(best<0||id<cands[best].id))){best=i;bestScore=value}}\n   if(best<0)throw Error(\'No eligible model-free candidates\');const id=cands[best].id;selected.add(id);mf.push(id);for(let j=0;j<n;j++)nearest[j]=Math.min(nearest[j],D[best*n+j]);\n }\n // Moving conditions within their method partition preserves both exact quotas and replicate blocks.\n const order=[...gp.map(id=>({id,method:\'gp\'})),...mf.map(id=>({id,method:\'model_free\'}))];\n for(const {slot,id,method} of fixedSlots.sort((a,b)=>a.slot-b.slot)){const found=order.findIndex(x=>x.id===id);if(found<0||order[found].method!==method)throw Error(\'Manual assignment has inconsistent quota\');[order[slot],order[found]]=[order[found],order[slot]]}\n const assignments=order.flatMap((o,s)=>Array.from({length:cfg.reps},(_,r)=>({well:expWells[s*cfg.reps+r],id:o.id,method:o.method,replicate:r+1,slot:s})));\n const warnings=partialSets(p,selected),coverage=nearest.reduce((a,b)=>a+b,0)/n;\n return {target,capacity,gpQuota:gq,mfQuota:mq,gpWells:gq*cfg.reps,mfWells:mq*cfg.reps,order,assignments,warnings,extensions,coverage,selected:[...selected].sort(),config:{...cfg},excluded:[...excluded],manualSlots:{...overrides},privacy:\'PRIVATE\',status:\'PROVISIONAL — verify sample replication, group integrity, and laboratory execution constraints\'};\n}\nfunction defaultConfig(){return {reps:4,quotaMode:\'percent\',gpPercent:50,gpWells:160,gpPolicy:\'published_designations\',objective:\'maximin\',geoWeight:.5,gpYield:.6,gpCost:.25,gpSD:.15}}\nfunction state(){return {input:null,cfg:defaultConfig(),manual:Object.fromEntries(PROTEINS.map(p=>[p,{slots:{},excluded:[]}])),plans:{},cache:{},active:null,search:\'\',error:\'\',showAllWarnings:false,sourceName:\'\'}}\nconst stats=p=>`<span class="dl-number">${p.capacity}</span> formulations · <b>${p.gpQuota}</b> GP / <b>${p.mfQuota}</b> model-free · ${p.gpWells}+${p.mfWells} wells`;\nfunction plateHTML(target,p,st){let lookup=Object.fromEntries(p.assignments.map(a=>[a.well,a])),nWarn=p.warnings.filter(w=>w.severity!==\'info\').length;\n let grid=\'<div class="dl-plate-grid"><span class="dl-corner"></span>\'+Array.from({length:24},(_,i)=>`<span class="dl-colhead ${i<4?\'reserved\':\'\'}">${i+1}</span>`).join(\'\');\n for(const r of ROWS){grid+=`<span class="dl-rowhead">${r}</span>`;for(let col=1;col<=24;col++){const well=r+String(col).padStart(2,\'0\'),a=lookup[well],kind=col<=4?\'reserved\':a?.method===\'gp\'?\'gp\':\'free\',sel=st.active?.target===target&&st.active?.well===well?\' active\':\'\',fixed=a&&st.manual[target].slots[a.slot]?\' pinned\':\'\';\n const title=col<=4?`${well}: reserved for standards/controls`:a?`${well} · ${a.id} · ${a.method} · replicate ${a.replicate}/${st.cfg.reps}`:`${well}: empty`;\n grid+=`<button type="button" class="dl-well ${kind}${sel}${fixed}" data-well="${well}" data-target="${target}" title="${esc(title)}" ${col<=4?\'disabled\':\'\'} aria-label="${esc(title)}"></button>`}}\n grid+=\'</div>\';\n const warn=p.warnings.filter(w=>w.kind===\'submitted_batch\'||w.kind===\'source_batch_incomplete\').length;\n return `<section class="dl-plate" data-protein="${target}"><div class="dl-plate-header"><div><div class="dl-eyebrow">384-WELL ARRAY · 16 × 24</div><h3>${target}</h3></div><div class="dl-health">${warn?\'⚠ \'+warn+\' partial batches\':\'✓ no explicit batch splits\'}</div></div><div class="dl-plate-meta">${stats(p)}</div>${grid}<div class="dl-plate-foot"><span><i class="dl-key gp"></i> GP-informed</span><span><i class="dl-key free"></i> Model-free</span><span><i class="dl-key reserved"></i> Reserved 1–4</span><span><i class="dl-key pinned"></i> Manual</span></div><div class="dl-submetric">Mean nearest-selected distance <b>${p.coverage.toFixed(3)}</b> · ${nWarn} incomplete batch/sweep warnings · ${p.extensions.length} GP forecast extensions</div></section>`}\nfunction selectedListHTML(st){return `<section class="dl-selection-panel"><div class="dl-list-title"><div><div class="dl-eyebrow">AUDITABLE OUTPUT</div><h3>Suggested submissions <span>· immutable record IDs</span></h3></div><p>Choose any submission to edit its plate position or exclude it from the selection.</p></div><div class="dl-selection-trio">${PROTEINS.map(t=>{const p=st.plans[t],ref=Object.fromEntries(st.input.targets[t].candidates.map(c=>[c.id,c]));return `<div class="dl-selected-protein"><h4>${t} <span>${p.order.length} formulations</span></h4><div class="dl-selected-scroll">${p.order.map((x,i)=>{const c=ref[x.id],well=p.assignments[i*st.cfg.reps].well,origin=x.method===\'gp\'?\'GP\':\'Model-free\';return `<div class="dl-selected-row"><button data-target="${t}" data-well="${well}" title="View ${esc(x.id)} at ${well}"><span class="dl-selected-source ${x.method}">${origin}</span><span><strong>${esc(c.sampleId||x.id)}</strong><small>${esc(x.id)} · ${well}</small></span></button><button class="dl-remove" data-action="excludeId" data-id="${esc(x.id)}" data-target="${t}" title="Exclude from suggestion">×</button></div>`}).join(\'\')}</div></div>`}).join(\'\')}</div></section>`}\nfunction warningsHTML(st){const all=PROTEINS.flatMap(t=>(st.plans[t]?.warnings||[]).map(w=>({...w,target:t}))),priority={high:0,medium:1,info:2};all.sort((a,b)=>priority[a.severity]-priority[b.severity]||a.target.localeCompare(b.target));const visible=st.showAllWarnings?all:all.slice(0,14);return `<div class="dl-warnings-head"><h3>Integrity & cohort warnings <span>${all.length}</span></h3><p>Only explicit metadata batches are treated as original submission batches. GP clusters and sweeps are advisory families; not every partial cluster is a protocol violation.</p></div><div class="dl-warning-list">${visible.map(w=>`<article class="dl-warning ${w.severity}"><span class="dl-warning-type">${w.kind.replaceAll(\'_\',\' \')}</span><b>${esc(w.target)} · ${esc(w.name)}</b><span>${w.nSelected}/${w.total} selected · ${w.missing.length} omitted${w.sourceMissing?\' · \'+w.sourceMissing+\' missing from source\':\'\'}</span><details><summary>Inspect missing IDs</summary><code>${esc(w.missing.join(\', \'))}</code></details></article>`).join(\'\')||\'<p>No partial groups or batches detected.</p>\'}</div>${all.length>14?`<button class="dl-link" data-action="toggleWarnings">${st.showAllWarnings?\'Show fewer\':\'Show all \'+all.length+\' warnings\'}</button>`:\'\'}`}\nfunction editorHTML(st){const a=st.active;if(!a)return `<div class="dl-editor-empty"><div class="dl-icon">◎</div><h3>Live plate editor</h3><p>Click any colored experimental well to replace or exclude its formulation. All ${st.cfg.reps} replicates update together and quotas remain enforced.</p></div>`;\n const p=st.plans[a.target],record=p?.assignments.find(x=>x.well===a.well),candidate=st.input.targets[a.target].candidates.find(x=>x.id===record?.id),q=st.search.trim().toLowerCase();const suggestions=st.input.targets[a.target].candidates.filter(c=>!q||[c.id,c.sampleId,c.author,...c.aliases].some(v=>String(v||\'\').toLowerCase().includes(q))).slice(0,12);\n return `<div class="dl-editor"><div class="dl-eyebrow">EDITOR · ${a.target} · ${a.well}</div><h3>${esc(candidate?.sampleId||record?.id||\'Select formulation\')}</h3><div class="dl-editor-metadata"><span>Slot ${record?.slot+1} of ${p.capacity}</span><span>${record?.method===\'gp\'?\'GP share\':\'Model-free share\'}</span><span>${st.cfg.reps} replicates</span></div><label class="dl-field">Search exact ID, sample or author<input id="dl-search" placeholder="Filter ${st.input.targets[a.target].n_distinct} distinct candidates" value="${esc(st.search)}" autocomplete="off"></label><div class="dl-matches">${suggestions.map(c=>`<button class="dl-match" data-action="replace" data-id="${esc(c.id)}"><span><strong>${esc(c.sampleId||c.id)}</strong><small>${esc(c.author||\'\')} · ${esc(c.id)}</small></span><span class="dl-match-arrow">↗</span></button>`).join(\'\')||\'<p>No matching submission</p>\'}</div><div class="dl-editor-actions"><button data-action="unpin">Restore suggested slot</button><button data-action="exclude">Exclude selected formulation</button></div><p class="dl-editor-hint">Replacing a formulation creates a manual lock in this slot’s strategy. Duplicates are not permitted; existing assignments may be exchanged automatically.</p></div>`}\nfunction configHTML(st){const c=st.cfg,available=st.input?PROTEINS.every(t=>st.input.targets[t].source_has_gp_annotations):false;return `<div class="dl-controls"><div class="dl-control-top"><div class="dl-eyebrow">EXPERIMENT DESIGN CONTROLS</div><h2>Decision parameters</h2><p>Mix two independent selection trajectories under exact physical-well constraints.</p></div>\n <label class="dl-field"><span>Technical replicates / formulation</span><select data-cfg="reps">${[1,2,4,5,8,10,16].map(v=>`<option value="${v}" ${c.reps==v?\'selected\':\'\'}>${v} × — ${320/v} formulations / plate</option>`).join(\'\')}</select></label>\n <label class="dl-field"><span>Quota unit</span><select data-cfg="quotaMode"><option value="percent" ${c.quotaMode===\'percent\'?\'selected\':\'\'}>Percentage of unique formulations</option><option value="wells" ${c.quotaMode===\'wells\'?\'selected\':\'\'}>Physical experimental wells</option></select></label>\n ${c.quotaMode===\'percent\'?`<label class="dl-field"><span>GP-informed share <strong>${c.gpPercent}%</strong> · Model-free ${100-c.gpPercent}%</span><input type="range" data-cfg="gpPercent" min="0" max="100" step="1" value="${c.gpPercent}" ${available?\'\':\'disabled\'}></label>`:`<label class="dl-field"><span>GP-informed wells <strong>${c.gpWells}</strong> / 320</span><input data-cfg="gpWells" type="number" min="0" max="320" step="${c.reps}" value="${c.gpWells}" ${available?\'\':\'disabled\'}><small>Must be a multiple of ${c.reps}; the remaining wells go to model-free.</small></label>`}\n <div class="dl-divider"></div><label class="dl-field"><span>GP selection basis</span><select data-cfg="gpPolicy"><option value="published_designations" ${c.gpPolicy===\'published_designations\'?\'selected\':\'\'}>Published group/run designations, then forecast</option><option value="forecast_ranked" ${c.gpPolicy===\'forecast_ranked\'?\'selected\':\'\'}>Model-forecast ranking only</option></select></label>\n <div class="dl-weight-title">Model forecast weights <small>Rank-based surrogate, not calibrated utility</small></div>\n ${[[\'gpYield\',\'Predicted success\'],[\'gpCost\',\'Lower log cost / gram\'],[\'gpSD\',\'Model uncertainty\']].map(([k,label])=>`<label class="dl-field"><span>${label} <b>${Number(c[k]).toFixed(2)}</b></span><input type="range" data-cfg="${k}" min="0" max="1" step="0.05" value="${c[k]}"></label>`).join(\'\')}\n <div class="dl-divider"></div><label class="dl-field"><span>Model-free criterion</span><select data-cfg="objective"><option value="maximin" ${c.objective===\'maximin\'?\'selected\':\'\'}>Greedy Maximin (exploration)</option><option value="facility_location" ${c.objective===\'facility_location\'?\'selected\':\'\'}>Facility location (representation)</option><option value="balanced" ${c.objective===\'balanced\'?\'selected\':\'\'}>Weighted combination</option></select></label>\n ${c.objective===\'balanced\'?`<label class="dl-field"><span>Maximin weight <strong>${c.geoWeight.toFixed(2)}</strong></span><input type="range" data-cfg="geoWeight" min="0" max="1" step="0.05" value="${c.geoWeight}"></label>`:\'\'}\n <p class="dl-math">Model-free score: α·min<sub>s∈S</sub> d(x,s) + (1−α)·N<sup>−1</sup>Σ<sub>u</sub> max(0, d(u,S)−d(u,x)). <i>Volume-based proxy, not biological response.</i></p>\n <div class="dl-utilities"><button data-action="export">Export private plate plans (.json)</button><button data-action="csv">Export well assignments (.csv)</button><button data-action="reset">Reset manual edits</button></div></div>`}\nfunction render(st,el){const loaded=!!st.input;el.innerHTML=`<main class="dl-root"><header class="dl-hero"><div><div class="dl-eyebrow">HTGAA / CFPS · EXPERIMENTAL DESIGN</div><h1>Plate Decision Lab <span>01 / 03</span></h1><p>Auditable GP-informed and chemistry-space selection, with live 384-well allocation.</p></div><div class="dl-hero-actions"><label class="dl-upload">${loaded?\'Replace private input\':\'Load private decision input\'}<input type="file" id="dl-upload" accept=".json,application/json"></label><span class="dl-source">${loaded?esc(st.sourceName):\'No dataset loaded\'}</span></div></header>\n ${!loaded?`<section class="dl-intro">${st.error?`<p role="alert" class="dl-error">${esc(st.error)}</p>`:""}<div class="dl-intro-icon">◈</div><h2>Private input required</h2><p>Generate <code>cfps_decision_input_private.json</code> with the included Colab notebook, then load it here. The authenticated website route can load the same file from a protected server endpoint.</p><label class="dl-upload">Choose private input JSON<input type="file" id="dl-upload-secondary" accept=".json"></label><p>No participant records are embedded in this source-only release.</p></section>`:\n `<div class="dl-workspace"><aside class="dl-sidebar">${configHTML(st)}</aside><div class="dl-main">${st.error?`<div role="alert" class="dl-error">${esc(st.error)}</div>`:\'\'}<div class="dl-kpis"><div><b>${uniqCapacity(st.cfg)*3}</b><span>Unique plate conditions</span></div><div><b>960</b><span>Experimental physical wells</span></div><div><b>192</b><span>Reserved wells</span></div><div><b>${Object.values(st.plans).reduce((s,p)=>s+p.warnings.filter(w=>w.kind===\'submitted_batch\').length,0)}</b><span>Partially selected declared batches</span></div></div>\n <div class="dl-plate-trio">${PROTEINS.map(t=>st.plans[t]?plateHTML(t,st.plans[t],st):`<section class="dl-plate dl-error">${t}: ${esc(st.error)}</section>`).join(\'\')}</div>${selectedListHTML(st)}<div class="dl-bottom"><section class="dl-warning-panel">${warningsHTML(st)}</section><section class="dl-editor-panel">${editorHTML(st)}</section></div></div></div>`}\n <footer class="dl-footer">Planning only · selection objectives are geometric/model-based proxies · plate coordinate mapping is deterministic preview, not Ginkgo’s run-time randomized layout · no private inputs stored by this tool.</footer></main>`;\n const inp=el.querySelector(\'#dl-search\');if(inp){inp.addEventListener(\'input\',e=>{st.search=e.target.value;const pos=e.target.selectionStart;render(st,el);const next=el.querySelector(\'#dl-search\');next?.focus();next?.setSelectionRange(pos,pos)})}\n}\nfunction compute(st){st.error=\'\';st.plans={};if(!st.input)return;try{for(const t of PROTEINS){st.plans[t]=solve(st.input,t,st.cfg,st.manual,st.cache)}}catch(e){st.error=String(e.message||e);st.plans={};}}\nfunction download(name,data,type){const blob=new Blob([data],{type}),url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=name;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),1000)}\nfunction mount(el,opts={}){if(!el)throw Error(\'Container missing\');const st=state();const refresh=()=>{compute(st);render(st,el)},setData=(o,name)=>{if(o?.schema!==\'cfps-decision-input/v1\'||!PROTEINS.every(t=>o.targets?.[t]?.candidates?.length))throw Error(\'Expected cfps-decision-input/v1 with all three protein datasets\');st.input=o;st.sourceName=name;st.cache={};if(!PROTEINS.every(t=>o.targets[t].source_has_gp_annotations)){st.cfg.gpPercent=0;st.cfg.gpWells=0}refresh()};\n el.addEventListener(\'change\',async e=>{if(e.target.matches(\'input[type=file]\')){const file=e.target.files?.[0];if(!file)return;try{setData(JSON.parse(await file.text()),file.name)}catch(err){st.error=\'Input error: \'+err.message;render(st,el)}return}if(e.target.dataset.cfg){const k=e.target.dataset.cfg;st.cfg[k]=[\'reps\',\'gpPercent\',\'gpWells\'].includes(k)?Number(e.target.value):[\'geoWeight\',\'gpYield\',\'gpCost\',\'gpSD\'].includes(k)?Number(e.target.value):e.target.value;refresh()}});\n el.addEventListener(\'input\',e=>{if(e.target.dataset.cfg&&e.target.type===\'range\'){const k=e.target.dataset.cfg;st.cfg[k]=Number(e.target.value);refresh()}});\n el.addEventListener(\'click\',e=>{const well=e.target.closest(\'[data-well]\');if(well){st.active={target:well.dataset.target,well:well.dataset.well};st.search=\'\';render(st,el);return}const action=e.target.closest(\'[data-action]\');if(!action)return;const a=action.dataset.action;\n  if(a===\'toggleWarnings\'){st.showAllWarnings=!st.showAllWarnings;render(st,el);return}\n  if(a===\'excludeId\'){const t=action.dataset.target,id=action.dataset.id;for(const [s,v] of Object.entries(st.manual[t].slots))if(v===id)delete st.manual[t].slots[s];if(!st.manual[t].excluded.includes(id))st.manual[t].excluded.push(id);refresh();return}\n  if(a===\'reset\'){for(const t of PROTEINS)st.manual[t]={slots:{},excluded:[]};st.active=null;refresh();return}\n  if(a===\'export\' || a===\'csv\'){if(st.error||PROTEINS.some(t=>!st.plans[t]))return;const body={schema:\'cfps-plate-decision-export/v1\',createdAt:new Date().toISOString(),sourceName:st.sourceName,sourceProvenance:st.input.provenance,clusterParity:st.input.cluster_parity,privacy:\'PRIVATE · DO NOT COMMIT\',plans:st.plans};if(a===\'export\'){download(\'cfps_decision_plans_PRIVATE.json\',JSON.stringify(body,null,2),\'application/json\');return}const rows=[\'protein,well,sample_id,method,replicate,slot,provisional\'];for(const t of PROTEINS)for(const x of st.plans[t].assignments)rows.push([t,x.well,x.id,x.method,x.replicate,x.slot+1,\'true\'].join(\',\'));download(\'cfps_decision_well_map_PRIVATE.csv\',rows.join(\'\\n\')+\'\\n\',\'text/csv\');return}\n  if(!st.active)return;const t=st.active.target,p=st.plans[t],rec=p?.assignments.find(x=>x.well===st.active.well);if(!rec)return;\n  if(a===\'replace\'){let id=action.dataset.id;const existing=Object.entries(st.manual[t].slots).find(([slot,v])=>v===id);if(existing)delete st.manual[t].slots[existing[0]];st.manual[t].slots[rec.slot]=id;st.manual[t].excluded=st.manual[t].excluded.filter(x=>x!==id);refresh();return}\n  if(a===\'unpin\'){delete st.manual[t].slots[rec.slot];refresh();return}\n  if(a===\'exclude\'){const id=rec.id;for(const [s,v] of Object.entries(st.manual[t].slots))if(v===id)delete st.manual[t].slots[s];if(!st.manual[t].excluded.includes(id))st.manual[t].excluded.push(id);refresh();return}\n });\n render(st,el);if(opts.data){try{setData(opts.data,\'in-memory input\')}catch(e){st.error=e.message;render(st,el)}}else if(opts.dataUrl){fetch(opts.dataUrl,{credentials:\'same-origin\',cache:\'no-store\'}).then(r=>{if(!r.ok)throw Error(`Authenticated decision endpoint returned ${r.status}`);return r.json()}).then(d=>setData(d,\'authenticated server source\')).catch(e=>{st.error=e.message;render(st,el)})}\n return {setData,refresh,getState:()=>st,solve:(target)=>solve(st.input,target,st.cfg,st.manual,st.cache)};\n}\nroot.CFPSDecisionLab={mount,solve,defaultConfig,distanceMatrix,partialSets,experimentalWells:expWells};\n})(typeof window!==\'undefined\'?window:globalThis);\n'
print("Embedded client asset:",'repo_additions/static/cfps/decision/decision.js',"length",len(viewer_js))

In [ ]:
viewer_css=':root{color-scheme:dark}.dl-root{--bg:#07101a;--pane:#101c29;--line:#273949;--muted:#92a3b7;--text:#e9f0f9;--gp:#4db6db;--free:#d3ac71;--bad:#e9a26b;font:14px/1.5 Inter,ui-sans-serif,system-ui,-apple-system,Segoe UI,sans-serif;background:var(--bg);color:var(--text);min-height:100vh;padding:24px 26px;box-sizing:border-box}.dl-root *{box-sizing:border-box}.dl-root button,.dl-root input,.dl-root select{font:inherit}.dl-root button{cursor:pointer}.dl-root h1,.dl-root h2,.dl-root h3,.dl-root p{margin:0}.dl-eyebrow{color:#91a8bd;letter-spacing:.18em;text-transform:uppercase;font-size:10px;font-weight:800}.dl-hero{display:flex;justify-content:space-between;align-items:center;gap:24px;border-bottom:1px solid var(--line);padding:5px 0 24px;margin-bottom:21px}.dl-hero h1{font-size:clamp(24px,2.45vw,38px);letter-spacing:-1.2px;font-weight:650;margin:5px 0}.dl-hero h1 span{font-size:12px;letter-spacing:.18em;color:#94a8bc;font-weight:500;vertical-align:middle}.dl-hero p,.dl-control-top p,.dl-intro p{color:var(--muted)}.dl-hero-actions{display:flex;align-items:flex-end;flex-direction:column;gap:8px}.dl-upload,.dl-utilities button{border:1px solid #527b9a;background:#1a3850;color:#fff;border-radius:8px;padding:9px 15px;font-weight:650;cursor:pointer;display:inline-block}.dl-upload input{display:none}.dl-source{color:var(--muted);font-size:11px;max-width:230px;overflow:hidden;text-overflow:ellipsis}.dl-workspace{display:grid;grid-template-columns:minmax(234px,282px) minmax(0,1fr);gap:18px;align-items:start}.dl-sidebar,.dl-warning-panel,.dl-editor-panel{background:var(--pane);border:1px solid var(--line);border-radius:12px}.dl-sidebar{position:sticky;top:14px;max-height:calc(100vh - 30px);overflow:auto}.dl-controls{padding:18px}.dl-control-top{margin-bottom:17px}.dl-control-top h2{font-size:20px;margin:4px 0}.dl-control-top p{font-size:12px}.dl-field{display:flex;flex-direction:column;gap:6px;margin:13px 0;color:#c5d1df;font-size:12px}.dl-field span{display:flex;justify-content:space-between;align-items:center;gap:6px}.dl-field b{color:#d7eafb}.dl-field input:not([type=range]),.dl-field select{background:#071521;color:#eaf2fb;border:1px solid #344a60;border-radius:7px;padding:8px;min-width:0;width:100%}.dl-field input[type=range]{width:100%;accent-color:#68bbd6;cursor:pointer}.dl-field small,.dl-weight-title small{font-size:10px;color:var(--muted)}.dl-divider{height:1px;background:#2b3b4b;margin:19px 0}.dl-weight-title{font-size:12px;font-weight:750;margin-top:20px}.dl-weight-title small{display:block;font-weight:400}.dl-math{font-size:10px;color:#9caec0;background:#0a1520;border:1px solid #2e4458;padding:10px;border-radius:6px;line-height:1.6}.dl-math i{display:block;color:#ba9878;margin-top:5px}.dl-utilities{display:flex;flex-direction:column;gap:8px;margin-top:20px}.dl-utilities button{font-size:11px;padding:8px}.dl-utilities button:last-child{border-color:#485467;background:#142331}.dl-main{min-width:0}.dl-kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin-bottom:16px}.dl-kpis>div{background:#112131;border:1px solid var(--line);border-radius:9px;padding:12px 14px}.dl-kpis b{font-size:25px;display:block;line-height:1.25;font-weight:620}.dl-kpis span{color:var(--muted);font-size:11px}.dl-plate-trio{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:11px}.dl-plate{min-width:0;background:#111f2c;border:1px solid #324557;border-radius:10px;padding:13px 11px 12px}.dl-plate-header{display:flex;justify-content:space-between;align-items:center;gap:8px}.dl-plate h3{font-size:23px;letter-spacing:-.4px;margin:2px 0}.dl-health{color:#9cbac8;font-size:10px;text-align:right;max-width:135px}.dl-plate-meta{color:#96afbf;border-bottom:1px solid #34495b;padding:8px 0 12px;font-size:10px;margin-bottom:10px}.dl-plate-meta b{color:#f0f6ff}.dl-number{color:#fff;font-weight:800}.dl-plate-grid{display:grid;grid-template-columns:14px repeat(24,minmax(0,1fr));column-gap:2px;row-gap:3px;align-items:center}.dl-colhead,.dl-rowhead{font-size:8px;text-align:center;color:#8aa3b9;font-weight:650}.dl-colhead.reserved{color:#6685a6}.dl-well{display:block;border:0;border-radius:2px;min-width:0;width:100%;height:12px;box-shadow:inset 0 0 0 .4px rgba(0,0,0,.1);padding:0;outline-offset:2px;transition:filter .08s}.dl-well:hover:not(:disabled){filter:brightness(1.5);outline:1px solid white;z-index:2}.dl-well.gp{background:#49a8cf}.dl-well.free{background:#c29a62}.dl-well.reserved{background:#354d68;opacity:.6;cursor:default}.dl-well.pinned{box-shadow:inset 0 0 0 2px #f6f7ef}.dl-well.active{outline:2px solid white;outline-offset:1px;z-index:3}.dl-plate-foot{display:flex;flex-wrap:wrap;gap:4px 10px;font-size:9px;color:#99afc1;margin-top:12px}.dl-plate-foot span{white-space:nowrap}.dl-key{display:inline-block;width:8px;height:8px;border-radius:2px;margin-right:3px}.dl-key.gp{background:var(--gp)}.dl-key.free{background:var(--free)}.dl-key.reserved{background:#354d68}.dl-key.pinned{border:2px solid white}.dl-submetric{font-size:10px;color:#9fb2c2;margin-top:13px}.dl-submetric b{color:#e6f0f7}.dl-bottom{display:grid;grid-template-columns:minmax(0,1.24fr) minmax(270px,.76fr);gap:15px;margin-top:16px}.dl-warning-panel,.dl-editor-panel{min-height:310px;padding:18px}.dl-warnings-head h3{font-size:17px}.dl-warnings-head h3 span{font-size:12px;background:#35455b;border-radius:15px;padding:2px 7px;margin-left:4px}.dl-warnings-head p{font-size:11px;color:var(--muted);margin:7px 0 16px}.dl-warning-list{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:7px}.dl-warning{padding:9px 10px;border:1px solid #33485b;border-left:3px solid #516a8a;border-radius:6px;background:#0b1925;min-width:0}.dl-warning.high{border-left-color:#e98e65}.dl-warning.medium{border-left-color:#c9a16a}.dl-warning b{font-size:11px;display:block;overflow-wrap:anywhere}.dl-warning>span:not(.dl-warning-type){font-size:10px;color:#92a8bc;display:block}.dl-warning-type{text-transform:uppercase;letter-spacing:.12em;font-size:8px;color:#b1beca}.dl-warning details{margin-top:4px;font-size:9px;color:#9cb1c8}.dl-warning code{display:block;overflow-wrap:anywhere;max-height:50px;overflow:auto}.dl-link{background:transparent;color:#8fc5e5;border:0;padding:10px 0;text-decoration:underline}.dl-editor-empty{display:flex;flex-direction:column;align-items:flex-start;justify-content:center;height:250px}.dl-editor-empty .dl-icon{font-size:34px;color:#95b8d0}.dl-editor-empty h3,.dl-editor h3{font-size:18px;margin:6px 0}.dl-editor-empty p{color:var(--muted);font-size:12px}.dl-editor-metadata{display:flex;flex-wrap:wrap;gap:6px;margin:8px 0}.dl-editor-metadata span{background:#1b3548;color:#a9c9de;border-radius:12px;padding:3px 8px;font-size:10px}.dl-matches{max-height:200px;overflow-y:auto;border:1px solid #344a60;border-radius:7px}.dl-match{width:100%;display:flex;align-items:center;justify-content:space-between;gap:8px;background:#0c1b29;border:0;border-bottom:1px solid #283d50;color:#e7f1fa;padding:8px 10px;text-align:left}.dl-match:hover{background:#244359}.dl-match strong,.dl-match small{display:block;font-size:10px;overflow-wrap:anywhere}.dl-match small{color:#91a5b9}.dl-match-arrow{color:#86cde5}.dl-editor-actions{display:flex;gap:7px;margin-top:10px;flex-wrap:wrap}.dl-editor-actions button{border:1px solid #49647a;color:#cadfee;border-radius:6px;background:#173044;font-size:10px;padding:7px}.dl-editor-hint{color:#8da6bc;font-size:10px;margin-top:12px}.dl-error{border:1px solid #9c6655;border-radius:7px;padding:12px;background:#3a2326;color:#ffcdc4;margin-bottom:13px}.dl-intro{max-width:640px;margin:7vh auto;text-align:center;border:1px solid #30465b;border-radius:14px;padding:48px 30px;background:#111e2b}.dl-intro-icon{font-size:55px;color:#90bbd3}.dl-intro h2{font-size:26px;margin:12px 0}.dl-intro p{margin:14px 0 22px}.dl-footer{color:#7890a6;font-size:10px;padding:22px 4px 6px;border-top:1px solid #2e4357;margin-top:22px}\n@media(min-width:1800px){.dl-well{height:15px}.dl-root{padding:30px 45px}.dl-plate{padding:19px}}@media(max-width:1370px){.dl-plate-trio{grid-template-columns:repeat(2,minmax(0,1fr))}.dl-plate:last-child{grid-column:1/-1;max-width:550px}}@media(max-width:1050px){.dl-workspace{grid-template-columns:1fr}.dl-sidebar{position:static;max-height:none}.dl-controls{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));column-gap:25px}.dl-control-top{grid-column:1/-1}.dl-bottom{grid-template-columns:1fr}}@media(max-width:700px){.dl-root{padding:13px}.dl-hero{flex-direction:column;align-items:flex-start}.dl-hero-actions{align-items:flex-start}.dl-controls{display:block}.dl-kpis{grid-template-columns:repeat(2,1fr)}.dl-plate-trio{grid-template-columns:1fr}.dl-plate:last-child{grid-column:auto;max-width:none}.dl-warning-list{grid-template-columns:1fr}}\n\n.dl-selection-panel{margin-top:16px;background:#101c29;border:1px solid #273949;border-radius:10px;padding:17px}.dl-list-title{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:14px}.dl-list-title h3{font-size:18px;margin:4px 0}.dl-list-title h3 span{font-size:11px;color:#8da8bc;font-weight:400}.dl-list-title p{font-size:11px;color:#91a8bd;max-width:330px}.dl-selection-trio{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px}.dl-selected-protein{border:1px solid #324557;border-radius:7px;background:#0b1724;min-width:0}.dl-selected-protein h4{font-size:13px;margin:0;padding:9px 11px;border-bottom:1px solid #2c4052}.dl-selected-protein h4 span{float:right;font-size:10px;color:#94a9bb;font-weight:400}.dl-selected-scroll{height:235px;overflow-y:auto}.dl-selected-row{display:flex;align-items:stretch;border-bottom:1px solid #233448;min-width:0}.dl-selected-row>button:first-child{background:transparent;color:#e8f0f8;display:flex;align-items:center;gap:7px;text-align:left;border:0;flex:1;min-width:0;padding:7px}.dl-selected-row>button:first-child:hover{background:#203548}.dl-selected-source{border-radius:3px;font-size:9px;font-weight:750;display:inline-block;min-width:45px;text-align:center;flex:none;padding:2px 4px}.dl-selected-source.gp{background:#18465c;color:#9fe0f4}.dl-selected-source.model_free{background:#51402b;color:#efc68a}.dl-selected-row strong{display:block;font-size:10px;font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;max-width:175px}.dl-selected-row small{font-size:9px;color:#8ba1b4;display:block;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;max-width:175px}.dl-remove{background:transparent;color:#ae8c89;border:0;padding:0 10px;font-size:17px}.dl-remove:hover{background:#432a33;color:#fff}\n@media(max-width:820px){.dl-selection-trio{grid-template-columns:1fr}.dl-list-title{display:block}.dl-selected-scroll{height:145px}}\n'
print("Embedded client asset:",'repo_additions/static/cfps/decision/decision.css',"length",len(viewer_css))

In [ ]:
from IPython.display import HTML,display
input_json=(PRIVATE/'cfps_decision_input_private.json').read_text()
# JSON safely embedded as <script type=application/json>; encoded angle bracket avoids script-tag injection.
input_json=input_json.replace('<','\\u003c').replace('>','\\u003e').replace('&','\\u0026')
page=f"""<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>CFPS Decision Lab · Private</title><style>{viewer_css}</style></head>
<body style="margin:0"><div id="cfps-root"></div><script type="application/json" id="private-data">{input_json}</script>
<script>{viewer_js}</script><script>window.CFPSDecisionLab.mount(document.getElementById('cfps-root'),{{data:JSON.parse(document.getElementById('private-data').textContent)}});</script></body></html>"""
(PRIVATE/'cfps_decision_live_preview_PRIVATE.html').write_text(page,encoding='utf8')
print('Preview saved:',PRIVATE/'cfps_decision_live_preview_PRIVATE.html')
print('Open the downloaded HTML locally for manual editing, then download edited plans from its Export controls.')


## 7 · Download private results (do NOT commit)

Includes the normalized decision input, machine-readable selected IDs, a 960-row experimental-well CSV across the three plates, full warnings, input hash, and a self-contained interactive HTML viewer. **No live Keycloak secret is needed for local notebook execution.**

In [ ]:
archive=Path('/content/cfps_decision_results_PRIVATE.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(PRIVATE.iterdir()):
        if path.is_file():z.write(path,arcname=path.name)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None,'ZIP verification failed'
print('Saved private results:',archive,'bytes:',archive.stat().st_size)
files.download(str(archive))
